LIVE TRADING - DAILY RAAM POSITIONS
------------------------------------

Run this notebook every day after the market close to refresh prices and get next-session BUY / HOLD / SELL / REBALANCE actions.

- Uses the same RAAM logic: ensemble positions, RSI ranks on cumulative Buy & Hold returns, and top-N allocation.
- Position signals are applied at the start of each ticker's own session (europe / na / crypto), not after an extra session delay.
- Final cell maps the latest ranks/positions to next-session target weights (not the lagged last-bar holdings).

---

In [1]:
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings

In [2]:
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
# List of tickers to download
TICKERS = [
    "BTC-USD",
    "TQQQ",
    "UPRO",
    "SMH.L",
    "QTUM",
    "GLD", 
]

START_DATE = '2021-01-01'

# Validate config shape (a bare string would iterate character-by-character)
if isinstance(TICKERS, str):
    raise ValueError("TICKERS must be a list of ticker symbols, not a single string.")
if not isinstance(TICKERS, (list, tuple)):
    raise ValueError(f"TICKERS must be a list of ticker symbols; got {type(TICKERS).__name__}.")

# Normalize once: strip, drop blanks, preserve order, remove duplicates
TICKERS = list(dict.fromkeys(str(t).strip() for t in TICKERS if str(t).strip()))
if not TICKERS:
    raise ValueError("TICKERS is empty. Provide at least one ticker symbol.")

START_DATE = str(START_DATE).strip()
try:
    start_ts = pd.Timestamp(START_DATE)
except Exception as exc:
    raise ValueError(f"START_DATE={START_DATE!r} is not a valid date.") from exc
if start_ts.normalize() > pd.Timestamp.today().normalize():
    raise ValueError(f"START_DATE={START_DATE!r} is in the future.")

# Download data from start date onwards (explicit column grouping for stable MultiIndex)
try:
    stock_data = yf.download(
        TICKERS,
        start=START_DATE,
        interval="1d",
        group_by="column",
        threads=False,
    )
except Exception as exc:
    raise RuntimeError(f"yfinance download failed for TICKERS={TICKERS!r}: {exc}") from exc

if stock_data is None or stock_data.empty:
    raise ValueError(f"Failed to download data for TICKERS={TICKERS!r} from yfinance")

def _select_close(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ("Close", ticker) in df.columns:
            return df[("Close", ticker)]
        if (ticker, "Close") in df.columns:
            return df[(ticker, "Close")]
        return None
    if len(TICKERS) == 1 and "Close" in df.columns:
        return df["Close"]
    return None

failed_tickers = []
for t in list(TICKERS):
    close_s = _select_close(stock_data, t)
    if close_s is None or close_s.dropna().empty:
        failed_tickers.append(t)

if failed_tickers:
    print(
        f"WARNING: dropping {len(failed_tickers)} ticker(s) with no usable Close data: "
        f"{failed_tickers}"
    )
    TICKERS = [t for t in TICKERS if t not in set(failed_tickers)]

if not TICKERS:
    raise ValueError(
        "No usable Close data for any requested ticker after download. "
        f"Failed: {failed_tickers!r}."
    )

# Bootstrap only: some helper cells still read a global TICKER name.
# Analysis loops every symbol in TICKERS (not just this one).
TICKER = TICKERS[0]

print(f"Successfully downloaded {len(stock_data)} records for {len(TICKERS)} tickers from {START_DATE}")
print(f"Tickers: {', '.join(TICKERS)}")
print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
print("\nFirst 5 rows:")
print(stock_data.head())

# Display the downloaded data
stock_data


[*********************100%***********************]  6 of 6 completed


Successfully downloaded 2107 records for 6 tickers from 2021-01-01
Tickers: BTC-USD, TQQQ, UPRO, SMH.L, QTUM, GLD
Data range: 2021-01-01 to 2026-10-08

First 5 rows:
Price              Close                                             \
Ticker           BTC-USD         GLD       QTUM    SMH.L       TQQQ   
Date                                                                  
2021-01-01  29374.152344         NaN        NaN      NaN        NaN   
2021-01-02  32127.267578         NaN        NaN      NaN        NaN   
2021-01-03  32782.023438         NaN        NaN      NaN        NaN   
2021-01-04  31971.914062  182.330002  39.613136  21.0375  20.668501   
2021-01-05  33992.429688  182.869995  40.352333  21.1775  21.186850   

Price                          High                                    ...  \
Ticker           UPRO       BTC-USD         GLD       QTUM      SMH.L  ...   
Date                                                                   ...   
2021-01-01        NaN  29600.62

Price              Close                                                 \
Ticker           BTC-USD         GLD        QTUM       SMH.L       TQQQ   
Date                                                                      
2021-01-01  29374.152344         NaN         NaN         NaN        NaN   
2021-01-02  32127.267578         NaN         NaN         NaN        NaN   
2021-01-03  32782.023438         NaN         NaN         NaN        NaN   
2021-01-04  31971.914062  182.330002   39.613136   21.037500  20.668501   
2021-01-05  33992.429688  182.869995   40.352333   21.177500  21.186850   
...                  ...         ...         ...         ...        ...   
2026-10-04  86480.304688         NaN         NaN         NaN        NaN   
2026-10-05  85786.593750  379.549988  156.399994  118.120003  83.120003   
2026-10-06  85557.562500  382.269989  157.389999  118.779999  84.250000   
2026-10-07           NaN  375.880005  154.440002         NaN  83.620003   
2026-10-08  82539.929688         NaN         NaN         NaN        NaN   

Price                           High                                      ...  \
Ticker            UPRO       BTC-USD         GLD        QTUM       SMH.L  ...   
Date                                                                      ...   
2021-01-01         NaN  29600.626953         NaN         NaN         NaN  ...   
2021-01-02         NaN  33155.117188         NaN         NaN         NaN  ...   
2021-01-03         NaN  34608.558594         NaN         NaN         NaN  ...   
2021-01-04   35.403030  33440.218750  182.399994   40.238607   21.424999  ...   
2021-01-05   36.104282  34437.589844  183.210007   40.371284   21.177500  ...   
...                ...           ...         ...         ...         ...  ...   
2026-10-04         NaN  86766.148438         NaN         NaN         NaN  ...   
2026-10-05  153.889999  86971.781250  380.779999  156.610001  118.832397  ...   
2026-10-06  156.410004  86664.539062  383.570007  158.330002  119.400002  ...   
2026-10-07  155.160004           NaN  378.250000  154.809998         NaN  ...   
2026-10-08         NaN  83427.117188         NaN         NaN         NaN  ...   

Price             Open                                           Volume  \
Ticker            QTUM       SMH.L       TQQQ        UPRO       BTC-USD   
Date                                                                      
2021-01-01         NaN         NaN        NaN         NaN  4.073030e+10   
2021-01-02         NaN         NaN        NaN         NaN  6.786542e+10   
2021-01-03         NaN         NaN        NaN         NaN  7.866524e+10   
2021-01-04   39.906919   21.040001  21.884714   37.338645  8.116348e+10   
2021-01-05   39.726859   21.177500  20.481849   35.196512  6.754732e+10   
...                ...         ...        ...         ...           ...   
2026-10-04         NaN         NaN        NaN         NaN  1.503165e+10   
2026-10-05  156.449997  118.239998  80.949997  150.869995  3.024265e+10   
2026-10-06  157.550003  118.360001  84.540001  155.820007  2.524597e+10   
2026-10-07  154.800003         NaN  82.320000  154.330002           NaN   
2026-10-08         NaN         NaN        NaN         NaN  3.690296e+10   

Price                                                                
Ticker             GLD      QTUM     SMH.L         TQQQ        UPRO  
Date                                                                 
2021-01-01         NaN       NaN       NaN          NaN         NaN  
2021-01-02         NaN       NaN       NaN          NaN         NaN  
2021-01-03         NaN       NaN       NaN          NaN         NaN  
2021-01-04  14331400.0   39500.0    4680.0  197472000.0  14195000.0  
2021-01-05  12718800.0   29600.0       0.0  118712800.0   8545800.0  
...                ...       ...       ...          ...         ...  
2026-10-04         NaN       NaN       NaN          NaN         NaN  
2026-10-05   4289400.0  345900.0  387078.0   36928700.0   20442

In [3]:
# TECHNICAL ANALYSIS INDICATORS USING TA-LIB

# Make sure stock_data / TICKERS are available from the download cell
if "stock_data" not in globals() or stock_data is None or getattr(stock_data, "empty", True):
    raise ValueError("stock_data is missing or empty. Run the download cell first.")
if "TICKERS" not in globals() or not TICKERS:
    raise ValueError("TICKERS is missing or empty. Run the download cell first.")

indicators_by_ticker = {}
_indicator_failures = []

# Helper indicator implementations (shared across tickers)
def validate_kama_params(period, fast_ema_constant, slow_ema_constant):
    try:
        period = int(period)
        fast_ema_constant = int(fast_ema_constant)
        slow_ema_constant = int(slow_ema_constant)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "KAMA parameters must be integers; "
            f"got period={period!r}, fast={fast_ema_constant!r}, slow={slow_ema_constant!r}"
        ) from exc
    if period <= 0 or fast_ema_constant <= 0 or slow_ema_constant <= 0:
        raise ValueError("KAMA parameters must be positive integers.")
    if fast_ema_constant >= slow_ema_constant:
        raise ValueError("KAMA fastest EMA constant must be less than slowest EMA constant.")
    return period, fast_ema_constant, slow_ema_constant

def _compute_kama_core(close, period, fast_ema_constant, slow_ema_constant):
    if close.dropna().shape[0] <= period:
        raise ValueError(f"KAMA requires more than {period} rows of close data")

    change = (close - close.shift(period)).abs()
    volatility = close.diff().abs().rolling(window=period).sum()
    er = change / (volatility + 1e-9)
    sc = (
        (er * (2 / (fast_ema_constant + 1) - 2 / (slow_ema_constant + 1)))
        + 2 / (slow_ema_constant + 1)
    ) ** 2
    sc = sc.clip(0, 1).fillna(0)

    kama_values = np.full(len(close), np.nan)
    seed = close.iloc[period]
    if pd.isna(seed):
        return pd.Series(kama_values, index=close.index)

    kama_values[period] = seed
    sc_values = sc.to_numpy(dtype=float)
    close_values = close.to_numpy(dtype=float)
    for i in range(period + 1, len(close)):
        if np.isnan(close_values[i]):
            kama_values[i] = np.nan
            continue
        prev = kama_values[i - 1]
        if np.isnan(prev):
            kama_values[i] = np.nan
            continue
        kama_values[i] = prev + sc_values[i] * (close_values[i] - prev)

    kama_values[:period] = np.nan
    return pd.Series(kama_values, index=close.index)

def compute_kama_series(close, period, fast_ema_constant, slow_ema_constant, index=None):
    period, fast_ema_constant, slow_ema_constant = validate_kama_params(
        period,
        fast_ema_constant,
        slow_ema_constant,
    )
    if not isinstance(close, pd.Series):
        if index is None:
            raise ValueError("index is required when close is not a Series")
        close = pd.Series(close, index=index)
    close = close.astype(float)
    return _compute_kama_core(close, period, fast_ema_constant, slow_ema_constant)

def validate_supertrend_params(atr_length, factor):
    try:
        atr_length_num = float(atr_length)
        factor_num = float(factor)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "SUPERTREND parameters must be numeric; "
            f"got atr_length={atr_length!r}, factor={factor!r}"
        ) from exc
    if not np.isfinite(atr_length_num) or not np.isfinite(factor_num):
        raise ValueError("SUPERTREND ATR length and factor must be finite numbers.")
    if atr_length_num != int(atr_length_num):
        raise ValueError(
            f"SUPERTREND ATR length must be an integer; got atr_length={atr_length!r}"
        )
    atr_length = int(atr_length_num)
    # Reject silent float truncation (e.g. 2.7 -> 2) while still allowing 3.0
    if factor_num != int(factor_num):
        raise ValueError(
            "SUPERTREND factor must be a whole number for this grid "
            f"(sensitivity uses int grids); got factor={factor!r}"
        )
    factor = int(factor_num)
    if atr_length <= 0 or factor <= 0:
        raise ValueError("SUPERTREND ATR length and factor must be positive integers.")
    return atr_length, factor

def _compute_supertrend_core(high, low, close, atr_length, factor):
    if len(high) != len(close) or len(low) != len(close):
        raise ValueError("High, Low, and Close arrays must have the same length for SUPERTREND")
    if len(close) == 0:
        raise ValueError("SUPERTREND requires a non-empty OHLC series")
    if len(close) <= atr_length:
        raise ValueError(f"SUPERTREND requires more than {atr_length} rows of OHLC data")

    high_arr = np.asarray(high, dtype=float)
    low_arr = np.asarray(low, dtype=float)
    close_arr = np.asarray(close, dtype=float)

    finite_mask = np.isfinite(high_arr) & np.isfinite(low_arr) & np.isfinite(close_arr)
    if not finite_mask.any():
        raise ValueError("SUPERTREND requires at least one finite OHLC row")
    if np.any(high_arr[finite_mask] < low_arr[finite_mask]):
        raise ValueError("SUPERTREND requires High >= Low on all finite rows")

    atr = talib.ATR(high_arr, low_arr, close_arr, timeperiod=atr_length)
    hl2 = (high_arr + low_arr) / 2.0
    basic_upper = hl2 + factor * atr
    basic_lower = hl2 - factor * atr

    final_upper = np.full(len(close_arr), np.nan)
    final_lower = np.full(len(close_arr), np.nan)
    supertrend = np.full(len(close_arr), np.nan)
    direction = np.full(len(close_arr), np.nan)

    for i in range(len(close_arr)):
        # Gap / bad bar: carry prior state forward so trend memory is not reset
        if (
            np.isnan(atr[i])
            or not np.isfinite(close_arr[i])
            or not np.isfinite(high_arr[i])
            or not np.isfinite(low_arr[i])
        ):
            if i > 0:
                final_upper[i] = final_upper[i - 1]
                final_lower[i] = final_lower[i - 1]
                direction[i] = direction[i - 1]
                supertrend[i] = supertrend[i - 1]
            continue

        if i == 0 or np.isnan(final_lower[i - 1]):
            final_lower[i] = basic_lower[i]
        elif basic_lower[i] > final_lower[i - 1] or close_arr[i - 1] < final_lower[i - 1]:
            final_lower[i] = basic_lower[i]
        else:
            final_lower[i] = final_lower[i - 1]

        if i == 0 or np.isnan(final_upper[i - 1]):
            final_upper[i] = basic_upper[i]
        elif basic_upper[i] < final_upper[i - 1] or close_arr[i - 1] > final_upper[i - 1]:
            final_upper[i] = basic_upper[i]
        else:
            final_upper[i] = final_upper[i - 1]

        # Seed first direction from bands, then prior-close momentum (avoid hl2 tie -> always +1)
        if i == 0 or np.isnan(direction[i - 1]):
            if close_arr[i] > final_upper[i]:
                direction[i] = 1.0
            elif close_arr[i] < final_lower[i]:
                direction[i] = -1.0
            elif i > 0 and np.isfinite(close_arr[i - 1]):
                direction[i] = 1.0 if close_arr[i] >= close_arr[i - 1] else -1.0
            else:
                direction[i] = 1.0 if close_arr[i] > hl2[i] else -1.0
        elif direction[i - 1] == 1.0:
            direction[i] = -1.0 if close_arr[i] < final_lower[i] else 1.0
        else:
            direction[i] = 1.0 if close_arr[i] > final_upper[i] else -1.0

        supertrend[i] = final_lower[i] if direction[i] == 1.0 else final_upper[i]

    return supertrend, direction

def compute_supertrend_series(high, low, close, atr_length, factor, index=None):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("SUPERTREND index length must match close length")
    supertrend, direction = _compute_supertrend_core(high, low, close, atr_length, factor)
    return (
        pd.Series(supertrend, index=index),
        pd.Series(direction, index=index),
    )

def validate_kalman_params(process_noise, measurement_noise):
    try:
        process_noise_num = float(process_noise)
        measurement_noise_num = float(measurement_noise)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "KALMAN parameters must be numeric; "
            f"got process_noise={process_noise!r}, measurement_noise={measurement_noise!r}"
        ) from exc
    if not np.isfinite(process_noise_num) or not np.isfinite(measurement_noise_num):
        raise ValueError("KALMAN process noise (Q) and measurement noise (R) must be finite numbers.")
    if process_noise_num != int(process_noise_num):
        raise ValueError(
            "KALMAN process noise (Q) must be a whole number for this grid "
            f"(sensitivity uses int grids); got process_noise={process_noise!r}"
        )
    if measurement_noise_num != int(measurement_noise_num):
        raise ValueError(
            "KALMAN measurement noise (R) must be a whole number for this grid "
            f"(sensitivity uses int grids); got measurement_noise={measurement_noise!r}"
        )
    process_noise = int(process_noise_num)
    measurement_noise = int(measurement_noise_num)
    if process_noise <= 0 or measurement_noise <= 0:
        raise ValueError("KALMAN process noise (Q) and measurement noise (R) must be positive integers.")
    return process_noise, measurement_noise

def _compute_kalman_core(close, process_noise, measurement_noise):
    close_arr = np.asarray(close, dtype=float)
    if close_arr.size == 0:
        raise ValueError("KALMAN requires a non-empty close series")
    if not np.isfinite(close_arr).any():
        raise ValueError("KALMAN requires at least one finite close price")

    kalman = np.full(close_arr.size, np.nan)
    q = float(process_noise)
    r = float(measurement_noise)

    first_idx = int(np.flatnonzero(np.isfinite(close_arr))[0])
    x = close_arr[first_idx]
    # Seed uncertainty from R so the first updates are not artificially overconfident
    p = float(measurement_noise)
    kalman[first_idx] = x

    for i in range(first_idx + 1, close_arr.size):
        z = close_arr[i]
        # Predict every bar (including gaps) so multi-bar holes inflate uncertainty
        p = p + q
        if not np.isfinite(z):
            # Carry estimate forward across gaps; do not update with bad measurements
            kalman[i] = x
            continue
        # Update: higher R trusts the filter prediction over noisy measurements
        k = p / (p + r)
        x = x + k * (z - x)
        p = (1.0 - k) * p
        kalman[i] = x

    return kalman

def compute_kalman_series(close, process_noise, measurement_noise, index=None):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("KALMAN index length must match close length")
    kalman = _compute_kalman_core(close, process_noise, measurement_noise)
    return pd.Series(kalman, index=index)

def validate_alma_params(length, offset, sigma):
    try:
        length_num = float(length)
        offset_num = float(offset)
        sigma_num = float(sigma)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "ALMA parameters must be numeric; "
            f"got length={length!r}, offset={offset!r}, sigma={sigma!r}"
        ) from exc
    if not np.isfinite(length_num) or not np.isfinite(offset_num) or not np.isfinite(sigma_num):
        raise ValueError("ALMA length, offset, and sigma must be finite numbers.")
    if length_num != int(length_num):
        raise ValueError(
            "ALMA length must be a whole number for this grid "
            f"(sensitivity uses int grids); got length={length!r}"
        )
    if offset_num != int(offset_num):
        raise ValueError(
            "ALMA offset must be a whole number for this grid "
            f"(sensitivity uses int grids; offset is hundredths, e.g. 85 = 0.85); "
            f"got offset={offset!r}"
        )
    if sigma_num != int(sigma_num):
        raise ValueError(
            "ALMA sigma must be a whole number for this grid "
            f"(sensitivity uses int grids); got sigma={sigma!r}"
        )
    length = int(length_num)
    offset = int(offset_num)
    sigma = int(sigma_num)
    if length <= 1:
        raise ValueError("ALMA length must be an integer greater than 1.")
    # 0..100 hundredths maps to classic ALMA distribution offset [0.0, 1.0]
    if offset < 0 or offset > 100:
        raise ValueError("ALMA offset must be an integer in 0..100 (hundredths of the distribution offset).")
    if sigma <= 0:
        raise ValueError("ALMA sigma must be a positive integer.")
    return length, offset, sigma

def _compute_alma_core(close, length, offset, sigma):
    # Defensive guards so direct callers cannot hit ZeroDivisionError / bad shapes.
    if length <= 1:
        raise ValueError("ALMA length must be an integer greater than 1.")
    if sigma <= 0:
        raise ValueError("ALMA sigma must be a positive integer.")
    if offset < 0 or offset > 100:
        raise ValueError("ALMA offset must be an integer in 0..100 (hundredths of the distribution offset).")

    close_arr = np.asarray(close, dtype=float)
    if close_arr.size == 0:
        raise ValueError("ALMA requires a non-empty close series")
    if close_arr.size < length:
        raise ValueError(f"ALMA requires at least {length} rows of close data")
    if not np.isfinite(close_arr).any():
        raise ValueError("ALMA requires at least one finite close price")

    offset_f = offset / 100.0
    m = int(np.floor(offset_f * (length - 1)))
    s = length / float(sigma)
    idxs = np.arange(length, dtype=float)
    weights = np.exp(-((idxs - m) ** 2) / (2.0 * s * s))
    norm = weights.sum()
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError("ALMA weights are degenerate for the selected parameters")

    # Vectorized weighted window (avoids Python per-bar loop on large grids).
    alma = np.full(close_arr.size, np.nan)
    windows = np.lib.stride_tricks.sliding_window_view(close_arr, window_shape=length)
    finite_mask = np.isfinite(windows).all(axis=1)
    if finite_mask.any():
        alma_vals = np.full(windows.shape[0], np.nan)
        alma_vals[finite_mask] = windows[finite_mask] @ weights / norm
        alma[length - 1:] = alma_vals

    return alma

def compute_alma_series(close, length, offset, sigma, index=None):
    length, offset, sigma = validate_alma_params(length, offset, sigma)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("ALMA index length must match close length")
    alma = _compute_alma_core(close, length, offset, sigma)
    return pd.Series(alma, index=index)

def _ohlcv_col(frame, field, ticker):
    if isinstance(frame.columns, pd.MultiIndex):
        if (field, ticker) in frame.columns:
            return frame[(field, ticker)].values
        if (ticker, field) in frame.columns:
            return frame[(ticker, field)].values
        raise KeyError(f"{field} not found for ticker {ticker!r} in stock_data columns")
    if field not in frame.columns:
        raise KeyError(f"{field} not found in stock_data columns")
    return frame[field].values

# LIVE TRADING: only indicator helper functions are needed.
# Per-ticker indicator DataFrames are unused by the RAAM allocation path.
required_indicator_helpers = [
    "validate_kama_params", "_compute_kama_core",
    "validate_supertrend_params", "_compute_supertrend_core",
    "validate_kalman_params", "_compute_kalman_core",
    "validate_alma_params", "_compute_alma_core",
]
missing_indicator_helpers = [n for n in required_indicator_helpers if n not in globals()]
if missing_indicator_helpers:
    raise NameError(
        "Indicator helpers failed to define: " + ", ".join(missing_indicator_helpers)
    )
print(
    "Indicator helpers ready (KAMA / SUPERTREND / KALMAN / ALMA). "
    "Skipped unused per-ticker indicator calculations."
)

Indicator helpers ready (KAMA / SUPERTREND / KALMAN / ALMA). Skipped unused per-ticker indicator calculations.


In [4]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

# Expect stock_data and TICKERS already exist
if "stock_data" not in globals() or stock_data is None or getattr(stock_data, "empty", True):
    raise ValueError("stock_data is missing or empty. Run the download cell first.")
if "TICKERS" not in globals() or not TICKERS:
    raise ValueError("TICKERS is missing or empty. Run the download cell first.")

def select_close_series(df, ticker):
    ticker = str(ticker).strip()
    if not ticker:
        raise ValueError("ticker is empty when selecting Close series.")
    if isinstance(df.columns, pd.MultiIndex):
        if ("Close", ticker) in df.columns:
            s = df[("Close", ticker)]
        elif (ticker, "Close") in df.columns:
            s = df[(ticker, "Close")]
        else:
            raise KeyError(
                f"Close not found for ticker {ticker!r} in stock_data MultiIndex columns."
            )
    else:
        if len(TICKERS) > 1:
            raise ValueError(
                "stock_data has flat columns but multiple TICKERS were requested; "
                "expected a MultiIndex from yfinance."
            )
        if "Close" not in df.columns:
            raise KeyError("Close not found in stock_data columns.")
        s = df["Close"]
    return s.astype(float).squeeze()

TRAIN_RATIO = 0.60
if not isinstance(TRAIN_RATIO, (int, float)) or not (0.0 < float(TRAIN_RATIO) < 1.0):
    raise ValueError(
        f"TRAIN_RATIO must be a number strictly between 0 and 1; got {TRAIN_RATIO!r}."
    )
TRAIN_RATIO = float(TRAIN_RATIO)

close_by_ticker = {}
train_close_by_ticker = {}
val_close_by_ticker = {}
_bars_per_year_by_ticker = {}
_calendar_days_by_ticker = {}
_has_weekend_bars_by_ticker = {}
_prepare_failures = []

for TICKER in TICKERS:
    try:
        close = select_close_series(stock_data, TICKER)
        close = close.replace([np.inf, -np.inf], np.nan).dropna()
        if close.empty or len(close) < 2:
            raise ValueError(f"Close series for TICKER={TICKER!r} is missing or too short.")
        if not isinstance(close.index, pd.DatetimeIndex):
            close.index = pd.to_datetime(close.index, errors="coerce")
        if close.index.isna().any() or len(close.index) < 2:
            raise ValueError(
                f"Close index for TICKER={TICKER!r} must be datetime-like with >= 2 valid timestamps."
            )
        close = close.sort_index()
        if close.index.has_duplicates:
            close = close[~close.index.duplicated(keep="last")]
        if close.empty or len(close) < 2:
            raise ValueError(f"Close series for TICKER={TICKER!r} is missing or too short.")
        close.name = "price"

        split_idx = int(len(close) * TRAIN_RATIO)
        if split_idx < 1 or split_idx >= len(close):
            raise ValueError(
                f"TRAIN_RATIO={TRAIN_RATIO!r} produced an invalid split for TICKER={TICKER!r} "
                f"(len={len(close)}, split_idx={split_idx})."
            )
        train_close = close.iloc[:split_idx].copy()
        val_close = close.iloc[split_idx:].copy()
        if len(train_close) < 2 or len(val_close) < 2:
            raise ValueError(
                f"train/val split too short for TICKER={TICKER!r} "
                f"(train={len(train_close)}, val={len(val_close)})."
            )

        _calendar_days = max((close.index[-1] - close.index[0]).days, 1)
        _bars_per_year = len(close) / (_calendar_days / 365.25)
        if not np.isfinite(_bars_per_year) or _bars_per_year <= 0:
            raise ValueError(
                f"Could not infer bars/year for TICKER={TICKER!r} "
                f"(bars={len(close)}, calendar_days={_calendar_days})."
            )
        _bars_per_year_by_ticker[TICKER] = float(_bars_per_year)
        _calendar_days_by_ticker[TICKER] = int(_calendar_days)
        # Weekend prints are the primary 24/7 signal. bars/year > 330 is only trusted
        # on long enough samples (short weekday stocks can exceed 330 bars/year).
        _has_weekend_bars_by_ticker[TICKER] = bool(
            np.isin(close.index.dayofweek, [5, 6]).any()
        )

        close_by_ticker[TICKER] = close
        train_close_by_ticker[TICKER] = train_close
        val_close_by_ticker[TICKER] = val_close

        print(
            f"{TICKER}: train={train_close.index[0].date()} -> {train_close.index[-1].date()} | "
            f"val={val_close.index[0].date()} -> {val_close.index[-1].date()}"
        )
    except Exception as exc:
        _prepare_failures.append(f"{TICKER}: {type(exc).__name__}: {exc}")
        print(f"SKIP prepare for {TICKER}: {type(exc).__name__}: {exc}")
        continue

if _prepare_failures:
    print(f"Prepare failures: {len(_prepare_failures)}")
    for _detail in _prepare_failures:
        print(f"  - {_detail}")

if not close_by_ticker:
    raise ValueError(f"Price preparation failed for every ticker in TICKERS={TICKERS!r}.")

# Keep TICKERS aligned to symbols that actually prepared successfully
TICKERS = [t for t in TICKERS if t in close_by_ticker]

# Auto-detect market calendar per ticker (mixed crypto/stock universes supported)
_MIN_CALENDAR_DAYS_FOR_BARS_HEURISTIC = 120
_is_crypto_by_ticker = {
    t: bool(
        _has_weekend_bars_by_ticker[t]
        or (
            _calendar_days_by_ticker[t] >= _MIN_CALENDAR_DAYS_FOR_BARS_HEURISTIC
            and _bars_per_year_by_ticker[t] > 330
        )
    )
    for t in close_by_ticker
}
trading_days_per_year_by_ticker = {
    t: (365 if flag else 252) for t, flag in _is_crypto_by_ticker.items()
}

FREQ = "1D"


def _apply_market_calendar(ticker):
    """Set shared calendar globals from a prepared ticker's detected market type."""
    global IS_24_7_MARKET, TRADING_DAYS_PER_YEAR, YEAR_FREQ
    ticker = str(ticker).strip()
    if ticker not in trading_days_per_year_by_ticker:
        raise KeyError(
            f"No market calendar prepared for ticker {ticker!r}. "
            f"Known: {sorted(trading_days_per_year_by_ticker)}"
        )
    IS_24_7_MARKET = bool(_is_crypto_by_ticker[ticker])
    TRADING_DAYS_PER_YEAR = int(trading_days_per_year_by_ticker[ticker])
    if TRADING_DAYS_PER_YEAR <= 0:
        raise ValueError(
            f"TRADING_DAYS_PER_YEAR for {ticker!r} must be > 0; got {TRADING_DAYS_PER_YEAR!r}"
        )
    YEAR_FREQ = f"{TRADING_DAYS_PER_YEAR}D"


class _CalendarSyncDict(dict):
    """Dict that applies the accessed ticker's market calendar to shared globals."""

    def __getitem__(self, key):
        if not dict.__contains__(self, key):
            raise KeyError(key)
        _apply_market_calendar(key)
        return dict.__getitem__(self, key)

    def get(self, key, default=None):
        if dict.__contains__(self, key):
            return self[key]
        return default


close_by_ticker = _CalendarSyncDict(close_by_ticker)


def sample_years(price_series):
    if price_series is None or len(price_series) < 2:
        raise ValueError("sample_years requires a price series with at least 2 rows.")
    if TICKER in trading_days_per_year_by_ticker:
        tdpy = trading_days_per_year_by_ticker[TICKER]
    else:
        # Infer from the series itself when TICKER is unavailable/stale.
        idx = pd.DatetimeIndex(pd.to_datetime(price_series.index, errors="coerce"))
        if idx.isna().any():
            raise ValueError("sample_years requires a datetime-like price index.")
        _calendar_days = max((idx[-1] - idx[0]).days, 1)
        _bars_per_year = len(price_series) / (_calendar_days / 365.25)
        _weekend = bool(np.isin(idx.dayofweek, [5, 6]).any())
        _dense_long = (
            _calendar_days >= _MIN_CALENDAR_DAYS_FOR_BARS_HEURISTIC
            and np.isfinite(_bars_per_year)
            and _bars_per_year > 330
        )
        tdpy = 365 if (_weekend or _dense_long) else 252
    if not np.isfinite(tdpy) or tdpy <= 0:
        raise ValueError(f"sample_years got invalid trading-days-per-year={tdpy!r}")
    return max(len(price_series) / float(tdpy), 1e-9)


# Keep single-frame aliases pointed at the first prepared ticker for any legacy refs
TICKER = TICKERS[0]
close = close_by_ticker[TICKER]
train_close = train_close_by_ticker[TICKER]
val_close = val_close_by_ticker[TICKER]

_calendar_summary = ", ".join(
    f"{t}:{'crypto/365D' if _is_crypto_by_ticker[t] else 'stock/252D'}" for t in TICKERS
)
print(
    f"Prepared {len(close_by_ticker)} tickers | "
    f"FREQ='{FREQ}' | calendars=[{_calendar_summary}]"
)


BTC-USD: train=2021-01-01 -> 2024-06-16 | val=2024-06-17 -> 2026-10-08
TQQQ: train=2021-01-04 -> 2024-06-14 | val=2024-06-17 -> 2026-10-07
UPRO: train=2021-01-04 -> 2024-06-14 | val=2024-06-17 -> 2026-10-07
SMH.L: train=2021-01-04 -> 2024-06-18 | val=2024-06-19 -> 2026-10-06
QTUM: train=2021-01-04 -> 2024-06-14 | val=2024-06-17 -> 2026-10-07
GLD: train=2021-01-04 -> 2024-06-14 | val=2024-06-17 -> 2026-10-07
Prepared 6 tickers | FREQ='1D' | calendars=[BTC-USD:crypto/365D, TQQQ:stock/252D, UPRO:stock/252D, SMH.L:stock/252D, QTUM:stock/252D, GLD:stock/252D]


In [5]:
# Define Indicator Grids
# LIVE TRADING: no parameter grid search. Ensemble params come from the leaderboard.
# Keep only signal builders + shared thresholds used by build_*_signals.

AROON_ENTRY_LEVEL = 49
STC_BUY_LEVEL = 50
STC_SELL_LEVEL = 50
RSI_CROSS_LEVEL = 50
ADX_TREND_LEVEL = 20
TRIX_CROSS_LEVEL = 0


def validate_triple_ema_params(ema1_period, ema2_period, ema3_period):
    ema1_period = int(ema1_period)
    ema2_period = int(ema2_period)
    ema3_period = int(ema3_period)
    if ema1_period <= 0 or ema2_period <= 0 or ema3_period <= 0:
        raise ValueError("EMA periods must be positive integers.")
    if not (ema1_period <= ema2_period <= ema3_period):
        raise ValueError("EMA periods must be ordered as EMA1 <= EMA2 <= EMA3.")
    return ema1_period, ema2_period, ema3_period


_ema_series_cache = {}


def get_ema_series(close_series, period):
    cache_key = (id(close_series), int(period))
    if cache_key not in _ema_series_cache:
        ema = vbt.MA.run(close_series, int(period), ewm=True).ma
        _ema_series_cache[cache_key] = pd.Series(ema.values.flatten(), index=close_series.index)
    return _ema_series_cache[cache_key]


def build_triple_ema_signals(close_series, params, shift_signals=True):
    ema1_period, ema2_period, ema3_period = validate_triple_ema_params(
        params["ema1_period"],
        params["ema2_period"],
        params["ema3_period"]
    )
    ema1 = get_ema_series(close_series, ema1_period)
    ema2 = get_ema_series(close_series, ema2_period)
    ema3 = get_ema_series(close_series, ema3_period)

    entries = (
        ema1.vbt.crossed_above(ema2) |
        ema1.vbt.crossed_above(ema3) |
        ema2.vbt.crossed_above(ema3)
    ).reindex(close_series.index).fillna(False)
    exits = (
        ema1.vbt.crossed_below(ema2) |
        ema1.vbt.crossed_below(ema3) |
        ema2.vbt.crossed_below(ema3)
    ).reindex(close_series.index).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool)
    )


def format_triple_ema_params(params):
    return f"EMA({params['ema1_period']},{params['ema2_period']},{params['ema3_period']})"


def validate_macd_params(fast_period, slow_period, signal_period):
    fast_period = int(fast_period)
    slow_period = int(slow_period)
    signal_period = int(signal_period)
    if fast_period <= 0 or slow_period <= 0 or signal_period <= 0:
        raise ValueError("MACD periods must be positive integers.")
    if fast_period >= slow_period:
        raise ValueError("MACD fast period must be less than slow period.")
    return fast_period, slow_period, signal_period


def compute_macd(close_series, fast_period, slow_period, signal_period):
    fast_period, slow_period, signal_period = validate_macd_params(
        fast_period,
        slow_period,
        signal_period
    )
    close_series = pd.Series(close_series, index=close_series.index).astype(float)
    if close_series.dropna().shape[0] < slow_period + signal_period:
        raise ValueError("Not enough price bars to compute MACD for the selected periods.")

    macd_line_arr, macd_signal_arr, macd_hist_arr = talib.MACD(
        close_series.to_numpy(dtype=float),
        fastperiod=fast_period,
        slowperiod=slow_period,
        signalperiod=signal_period
    )
    return (
        pd.Series(macd_line_arr, index=close_series.index),
        pd.Series(macd_signal_arr, index=close_series.index),
        pd.Series(macd_hist_arr, index=close_series.index)
    )


def build_macd_signals(close_series, params, shift_signals=True):
    macd_line, macd_signal_line, macd_hist = compute_macd(
        close_series,
        params["macd_fast_period"],
        params["macd_slow_period"],
        params["macd_signal_period"]
    )
    entries = macd_line.vbt.crossed_above(macd_signal_line).reindex(close_series.index).fillna(False)
    exits = macd_line.vbt.crossed_below(macd_signal_line).reindex(close_series.index).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool)
    )


def format_macd_params(params):
    return (
        f"MACD({params['macd_fast_period']},"
        f"{params['macd_slow_period']},"
        f"{params['macd_signal_period']})"
    )


# AROON Up/Down crossover strategy (TA-Lib AROON timeperiod)
def validate_aroon_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"AROON timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("AROON timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "AROON timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("AROON timeperiod must be a positive integer.")
    return timeperiod


def validate_aroon_entry_level(level):
    try:
        level_num = float(level)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"AROON entry level must be numeric; got level={level!r}") from exc
    if not np.isfinite(level_num):
        raise ValueError("AROON entry level must be a finite number.")
    if not 0.0 <= level_num <= 100.0:
        raise ValueError(
            f"AROON entry level must be between 0 and 100 (Aroon scale); got {level_num}."
        )
    return level_num


AROON_ENTRY_LEVEL = validate_aroon_entry_level(AROON_ENTRY_LEVEL)


_aroon_ohlc_cache = {}
_aroon_series_cache = {}


def get_aroon_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError(
            "stock_data/TICKER are missing. Run the download cell first, then set "
            "TICKER to the active asset (multi-asset notebooks set TICKER inside "
            "the per-ticker loop) before OHLC-based indicators."
        )
    if "_select_ohlc_column" not in globals():
        raise ValueError(
            "Shared OHLC helper `_select_ohlc_column` is missing. "
            "Run cell 5 (Define Indicator Grids) fully (SUPERTREND section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("AROON close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _aroon_ohlc_cache:
        return _aroon_ohlc_cache[cache_key]

    price_index = pd.DatetimeIndex(close_series.index)
    if price_index.tz is not None:
        price_index = price_index.tz_localize(None)

    high = _select_ohlc_column(stock_data, TICKER, "High")
    low = _select_ohlc_column(stock_data, TICKER, "Low")
    if isinstance(high.index, pd.DatetimeIndex) and high.index.tz is not None:
        high = high.copy()
        high.index = high.index.tz_localize(None)
    if isinstance(low.index, pd.DatetimeIndex) and low.index.tz is not None:
        low = low.copy()
        low.index = low.index.tz_localize(None)

    high = high.reindex(price_index)
    low = low.reindex(price_index)
    close = pd.Series(
        pd.Series(close_series).astype(float).to_numpy(),
        index=price_index,
        name=getattr(close_series, "name", None),
    )

    finite_close = np.isfinite(close.to_numpy(dtype=float))
    if not finite_close.any():
        raise ValueError("AROON close_series contains no finite prices.")
    valid_mask = pd.Series(finite_close, index=close.index)
    high_vals = high.loc[valid_mask].to_numpy(dtype=float)
    low_vals = low.loc[valid_mask].to_numpy(dtype=float)
    if (not np.isfinite(high_vals).all()) or (not np.isfinite(low_vals).all()):
        raise ValueError(
            "High/Low data is missing or non-finite for one or more dates in the selected price series."
        )
    if (high_vals < low_vals).any():
        raise ValueError("AROON requires High >= Low on all valid close dates.")

    _aroon_ohlc_cache[cache_key] = (high, low, close)
    return _aroon_ohlc_cache[cache_key]


def compute_aroon(close_series, timeperiod):
    timeperiod = validate_aroon_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("AROON close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _aroon_series_cache:
        return _aroon_series_cache[cache_key]

    high, low, close = get_aroon_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= timeperiod:
        raise ValueError("Not enough price bars to compute AROON for the selected timeperiod.")

    try:
        aroon_down_arr, aroon_up_arr = talib.AROON(
            high.to_numpy(dtype=float),
            low.to_numpy(dtype=float),
            timeperiod=timeperiod,
        )
    except Exception as exc:
        raise ValueError(
            f"talib.AROON failed for timeperiod={timeperiod}: {type(exc).__name__}: {exc}"
        ) from exc

    aroon_up = pd.Series(aroon_up_arr, index=close.index)
    aroon_down = pd.Series(aroon_down_arr, index=close.index)
    if (~(aroon_up.isna() | aroon_down.isna())).sum() == 0:
        raise ValueError("AROON could not be computed for the selected parameter combination.")

    _aroon_series_cache[cache_key] = (aroon_up, aroon_down)
    return _aroon_series_cache[cache_key]


def build_aroon_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for AROON, got {type(params).__name__}.")
    missing_keys = [key for key in ("aroon_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"AROON params missing keys: {missing_keys}")
    if "AROON_ENTRY_LEVEL" not in globals():
        raise ValueError("AROON_ENTRY_LEVEL is missing. Re-run cell 5 (Define Indicator Grids).")
    entry_level = validate_aroon_entry_level(AROON_ENTRY_LEVEL)

    aroon_up, aroon_down = compute_aroon(close_series, params["aroon_timeperiod"])
    # Require both lines valid on this bar and the prior bar so TA-Lib warmup NaNs
    # do not register as false Up/Down crosses (same pattern as KAMA/RSI).
    valid = aroon_up.notna() & aroon_down.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Entry trigger: Up crosses above Down, and Up is strictly above the entry level.
    entries = (
        aroon_up.vbt.crossed_above(aroon_down)
        & (aroon_up > entry_level)
        & valid_cross
    ).fillna(False)
    exits = (aroon_down.vbt.crossed_above(aroon_up) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_aroon_params(params):
    return f"AROON({params['aroon_timeperiod']})"


def validate_aroon_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_aroon_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "AROON sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("AROON sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# STC Schaff Trend Cycle strategy (EMA short/long + cycle period)
def validate_stc_params(ema_short, ema_long, cycle_period):
    ema_short = int(ema_short)
    ema_long = int(ema_long)
    cycle_period = int(cycle_period)
    if ema_short <= 0 or ema_long <= 0 or cycle_period <= 0:
        raise ValueError("STC periods must be positive integers.")
    if ema_short >= ema_long:
        raise ValueError("STC short EMA period must be less than long EMA period.")
    return ema_short, ema_long, cycle_period


def compute_stc(close_series, ema_short, ema_long, cycle_period):
    ema_short, ema_long, cycle_period = validate_stc_params(ema_short, ema_long, cycle_period)
    close = np.asarray(close_series, dtype=float)
    stc_ema_short = talib.EMA(close, timeperiod=ema_short)
    stc_ema_long = talib.EMA(close, timeperiod=ema_long)
    stc_macd = stc_ema_short - stc_ema_long

    stc_macd_highest = talib.MAX(stc_macd, timeperiod=cycle_period)
    stc_macd_lowest = talib.MIN(stc_macd, timeperiod=cycle_period)
    stc_macd_denom = stc_macd_highest - stc_macd_lowest
    with np.errstate(divide="ignore", invalid="ignore"):
        stc_k_macd = np.where(
            np.abs(stc_macd_denom) > 0,
            100.0 * (stc_macd - stc_macd_lowest) / stc_macd_denom,
            0.0,
        )
    stc_d_macd = talib.EMA(stc_k_macd, timeperiod=cycle_period)

    stc_d_highest = talib.MAX(stc_d_macd, timeperiod=cycle_period)
    stc_d_lowest = talib.MIN(stc_d_macd, timeperiod=cycle_period)
    stc_d_denom = stc_d_highest - stc_d_lowest
    with np.errstate(divide="ignore", invalid="ignore"):
        stc = np.where(
            np.abs(stc_d_denom) > 0,
            100.0 * (stc_d_macd - stc_d_lowest) / stc_d_denom,
            0.0,
        )
    stc = pd.Series(stc, index=close_series.index).replace([np.inf, -np.inf], np.nan)
    return stc


def build_stc_signals_from_series(
    stc,
    buy_level=STC_BUY_LEVEL,
    sell_level=STC_SELL_LEVEL,
    shift_signals=True,
):
    entries = (stc > buy_level) & (stc.shift(1) <= buy_level)
    exits = (stc < sell_level) & (stc.shift(1) >= sell_level)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=stc.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=stc.index, dtype=bool),
    )


def build_stc_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for STC, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("stc_ema_short_period", "stc_ema_long_period", "stc_cycle_period")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"STC params missing keys: {missing_keys}")

    ema_short, ema_long, cycle_period = validate_stc_params(
        params["stc_ema_short_period"],
        params["stc_ema_long_period"],
        params["stc_cycle_period"],
    )
    stc = compute_stc(close_series, ema_short, ema_long, cycle_period)
    if stc.notna().sum() == 0:
        raise ValueError("STC could not be computed for the selected parameter combination.")
    return build_stc_signals_from_series(stc, shift_signals=shift_signals)


def format_stc_params(params):
    return (
        f"STC({params['stc_ema_short_period']},"
        f"{params['stc_ema_long_period']},"
        f"{params['stc_cycle_period']})"
    )


# KAMA price crossover strategy (custom KAMA: period + fastest/slowest EMA constants)
if (
    "validate_kama_params" not in globals()
    or "_compute_kama_core" not in globals()
):
    raise ValueError("KAMA helpers are missing. Run cell 3 (TA-Lib indicators) first.")


_kama_series_cache = {}


def get_kama_series(close_series, period, fast_ema_constant, slow_ema_constant):
    period, fast_ema_constant, slow_ema_constant = validate_kama_params(
        period,
        fast_ema_constant,
        slow_ema_constant,
    )
    cache_key = (id(close_series), period, fast_ema_constant, slow_ema_constant)
    if cache_key not in _kama_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        kama_line = _compute_kama_core(
            close,
            period,
            fast_ema_constant,
            slow_ema_constant,
        )
        if kama_line.notna().sum() == 0:
            raise ValueError("KAMA could not be computed for the selected parameter combination.")
        _kama_series_cache[cache_key] = (close, kama_line)
    return _kama_series_cache[cache_key]


def build_kama_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for KAMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("kama_period", "kama_fast_ema_constant", "kama_slow_ema_constant")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"KAMA params missing keys: {missing_keys}")

    close, kama_line = get_kama_series(
        close_series,
        params["kama_period"],
        params["kama_fast_ema_constant"],
        params["kama_slow_ema_constant"],
    )
    valid_kama = kama_line.notna()
    valid_cross = valid_kama & valid_kama.shift(1).fillna(False)
    entries = (close.vbt.crossed_above(kama_line) & valid_cross).fillna(False)
    exits = (close.vbt.crossed_below(kama_line) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_kama_params(params):
    return (
        f"KAMA({params['kama_period']},"
        f"{params['kama_fast_ema_constant']},"
        f"{params['kama_slow_ema_constant']})"
    )


# SUPERTREND trend-flip strategy (ATR length + factor/multiplier via TA-Lib ATR)
if (
    "validate_supertrend_params" not in globals()
    or "_compute_supertrend_core" not in globals()
):
    raise ValueError("SUPERTREND helpers are missing. Run cell 3 (TA-Lib indicators) first.")


_supertrend_ohlc_cache = {}
_supertrend_series_cache = {}


def _select_ohlc_column(df, ticker, column_name):
    if isinstance(df.columns, pd.MultiIndex):
        key = (column_name, ticker)
        if key in df.columns:
            series = df[key]
        else:
            # Exact level match only — avoid substring false positives like "HigherHigh"
            cols = [
                c for c in df.columns
                if isinstance(c, tuple) and len(c) >= 1 and c[0] == column_name
            ]
            if not cols:
                raise KeyError(f"{column_name} not found for ticker {ticker!r}")
            series = df[cols[0]]
    else:
        if column_name not in df.columns:
            raise KeyError(f"{column_name} not found in stock_data columns")
        series = df[column_name]
    return pd.Series(series.astype(float).to_numpy(), index=series.index, name=column_name)


def get_supertrend_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError(
            "stock_data/TICKER are missing. Run the download cell first, then set "
            "TICKER to the active asset (multi-asset notebooks set TICKER inside "
            "the per-ticker loop) before OHLC-based indicators."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("SUPERTREND close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _supertrend_ohlc_cache:
        return _supertrend_ohlc_cache[cache_key]

    high = _select_ohlc_column(stock_data, TICKER, "High").reindex(close_series.index)
    low = _select_ohlc_column(stock_data, TICKER, "Low").reindex(close_series.index)
    close = pd.Series(close_series, index=close_series.index).astype(float)

    valid_close = close.notna()
    if not valid_close.any():
        raise ValueError("SUPERTREND close_series contains no finite prices.")
    if high.loc[valid_close].isna().any() or low.loc[valid_close].isna().any():
        raise ValueError(
            "High/Low data is missing for one or more dates in the selected price series."
        )
    if (high.loc[valid_close] < low.loc[valid_close]).any():
        raise ValueError("SUPERTREND requires High >= Low on all valid close dates.")

    _supertrend_ohlc_cache[cache_key] = (high, low, close)
    return _supertrend_ohlc_cache[cache_key]


def get_supertrend_series(close_series, atr_length, factor):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    cache_key = (id(close_series), atr_length, factor)
    if cache_key not in _supertrend_series_cache:
        high, low, close = get_supertrend_ohlc(close_series)
        supertrend_line, direction = _compute_supertrend_core(
            high.to_numpy(dtype=float),
            low.to_numpy(dtype=float),
            close.to_numpy(dtype=float),
            atr_length,
            factor,
        )
        supertrend_line = pd.Series(supertrend_line, index=close.index)
        direction = pd.Series(direction, index=close.index)
        if direction.notna().sum() == 0:
            raise ValueError("SUPERTREND could not be computed for the selected parameter combination.")
        _supertrend_series_cache[cache_key] = (close, supertrend_line, direction)
    return _supertrend_series_cache[cache_key]


def build_supertrend_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for SUPERTREND, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("supertrend_atr_length", "supertrend_factor")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"SUPERTREND params missing keys: {missing_keys}")

    close, _supertrend_line, direction = get_supertrend_series(
        close_series,
        params["supertrend_atr_length"],
        params["supertrend_factor"],
    )
    valid_dir = direction.notna()
    valid_flip = valid_dir & valid_dir.shift(1).fillna(False)
    # Buy: close above red line → direction flips to green (+1)
    entries = ((direction == 1.0) & (direction.shift(1) == -1.0) & valid_flip).fillna(False)
    # Sell: close below green line → direction flips to red (-1)
    exits = ((direction == -1.0) & (direction.shift(1) == 1.0) & valid_flip).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_supertrend_params(params):
    return (
        f"SUPERTREND(ATR={params['supertrend_atr_length']},"
        f"Factor={params['supertrend_factor']})"
    )


def validate_supertrend_sensitivity_params(atr_length, factor, factor_anchor):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    if int(factor_anchor) != factor:
        raise ValueError("SUPERTREND sensitivity anchor must match the factor.")
    return atr_length, factor, factor


# KALMAN price vs filter strategy (Q=process noise, R=measurement noise)
if (
    "validate_kalman_params" not in globals()
    or "_compute_kalman_core" not in globals()
):
    raise ValueError("KALMAN helpers are missing. Run cell 3 (TA-Lib indicators) first.")


_kalman_series_cache = {}


def get_kalman_series(close_series, process_noise, measurement_noise):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if close_series is None or len(close_series) == 0:
        raise ValueError("KALMAN close_series is empty.")
    cache_key = (id(close_series), process_noise, measurement_noise)
    if cache_key not in _kalman_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        close_values = close.to_numpy(dtype=float)
        if not np.isfinite(close_values).any():
            raise ValueError("KALMAN close_series contains no finite prices.")
        kalman_line = pd.Series(
            _compute_kalman_core(close_values, process_noise, measurement_noise),
            index=close.index,
        )
        if kalman_line.notna().sum() == 0:
            raise ValueError("KALMAN could not be computed for the selected parameter combination.")
        _kalman_series_cache[cache_key] = (close, kalman_line)
    return _kalman_series_cache[cache_key]


def build_kalman_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for KALMAN, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("kalman_process_noise", "kalman_measurement_noise")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"KALMAN params missing keys: {missing_keys}")

    close, kalman_line = get_kalman_series(
        close_series,
        params["kalman_process_noise"],
        params["kalman_measurement_noise"],
    )
    valid = kalman_line.notna() & close.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: price closes above the Kalman filter line (cross into above)
    entries = (close.vbt.crossed_above(kalman_line) & valid_cross).fillna(False)
    # Sell: price closes below the Kalman filter line (cross into below)
    exits = (close.vbt.crossed_below(kalman_line) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_kalman_params(params):
    return (
        f"KALMAN(Q={params['kalman_process_noise']},"
        f"R={params['kalman_measurement_noise']})"
    )


def validate_kalman_sensitivity_params(process_noise, measurement_noise, measurement_noise_anchor):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if int(measurement_noise_anchor) != measurement_noise:
        raise ValueError("KALMAN sensitivity anchor must match the measurement noise (R).")
    return process_noise, measurement_noise, measurement_noise


# RSI midline crossover strategy (TA-Lib RSI timeperiod; buy/sell on cross of 50)
def validate_rsi_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"RSI timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("RSI timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "RSI timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("RSI timeperiod must be a positive integer.")
    return timeperiod


_rsi_series_cache = {}


def compute_rsi(close_series, timeperiod):
    timeperiod = validate_rsi_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("RSI close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _rsi_series_cache:
        return _rsi_series_cache[cache_key]

    close = pd.Series(close_series, index=close_series.index).astype(float)
    close_values = close.to_numpy(dtype=float)
    if not np.isfinite(close_values).any():
        raise ValueError("RSI close_series contains no finite prices.")
    if np.isfinite(close_values).sum() < timeperiod + 1:
        raise ValueError("Not enough price bars to compute RSI for the selected timeperiod.")

    rsi = pd.Series(
        talib.RSI(close_values, timeperiod=timeperiod),
        index=close.index,
    )
    if rsi.notna().sum() == 0:
        raise ValueError("RSI could not be computed for the selected parameter combination.")
    _rsi_series_cache[cache_key] = rsi
    return rsi


def build_rsi_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for RSI, got {type(params).__name__}.")
    missing_keys = [key for key in ("rsi_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"RSI params missing keys: {missing_keys}")

    rsi = compute_rsi(close_series, params["rsi_timeperiod"])
    valid = rsi.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: RSI crosses above 50 (rising bullish momentum)
    entries = (
        (rsi > RSI_CROSS_LEVEL) & (rsi.shift(1) <= RSI_CROSS_LEVEL) & valid_cross
    ).fillna(False)
    # Sell: RSI crosses below 50 (increasing bearish momentum)
    exits = (
        (rsi < RSI_CROSS_LEVEL) & (rsi.shift(1) >= RSI_CROSS_LEVEL) & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=rsi.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=rsi.index, dtype=bool),
    )


def format_rsi_params(params):
    return f"RSI({params['rsi_timeperiod']})"


def validate_rsi_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_rsi_params(timeperiod)
    if int(timeperiod_anchor_a) != timeperiod or int(timeperiod_anchor_b) != timeperiod:
        raise ValueError("RSI sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod



# ADX +DI/-DI crossover strategy (TA-Lib PLUS_DI / MINUS_DI; ADX = Wilder-smoothed DX)
def validate_adx_params(di_length, adx_smoothing):
    try:
        di_length_num = float(di_length)
        adx_smoothing_num = float(adx_smoothing)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"ADX params must be numeric; got di_length={di_length!r}, "
            f"adx_smoothing={adx_smoothing!r}"
        ) from exc
    if not np.isfinite(di_length_num) or not np.isfinite(adx_smoothing_num):
        raise ValueError("ADX DI length and ADX smoothing must be finite numbers.")
    if di_length_num != int(di_length_num):
        raise ValueError(
            "ADX DI length must be a whole number for this grid "
            f"(sensitivity uses int grids); got di_length={di_length!r}"
        )
    if adx_smoothing_num != int(adx_smoothing_num):
        raise ValueError(
            "ADX smoothing timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got adx_smoothing={adx_smoothing!r}"
        )
    di_length = int(di_length_num)
    adx_smoothing = int(adx_smoothing_num)
    if di_length <= 0 or adx_smoothing <= 0:
        raise ValueError("ADX DI length and ADX smoothing must be positive integers.")
    return di_length, adx_smoothing


def validate_adx_trend_level(level):
    try:
        level_num = float(level)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"ADX trend level must be numeric; got level={level!r}") from exc
    if not np.isfinite(level_num):
        raise ValueError("ADX trend level must be a finite number.")
    if not 0.0 <= level_num <= 100.0:
        raise ValueError(
            f"ADX trend level must be between 0 and 100 (ADX scale); got {level_num}."
        )
    return level_num


ADX_TREND_LEVEL = validate_adx_trend_level(ADX_TREND_LEVEL)


_adx_ohlc_cache = {}
_adx_di_cache = {}
_adx_series_cache = {}


def get_adx_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError(
            "stock_data/TICKER are missing. Run the download cell first, then set "
            "TICKER to the active asset (multi-asset notebooks set TICKER inside "
            "the per-ticker loop) before OHLC-based indicators."
        )
    if "_select_ohlc_column" not in globals():
        raise ValueError(
            "Shared OHLC helper `_select_ohlc_column` is missing. "
            "Run cell 5 (Define Indicator Grids) fully (SUPERTREND section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _adx_ohlc_cache:
        return _adx_ohlc_cache[cache_key]

    price_index = pd.DatetimeIndex(close_series.index)
    if price_index.tz is not None:
        price_index = price_index.tz_localize(None)

    high = _select_ohlc_column(stock_data, TICKER, "High")
    low = _select_ohlc_column(stock_data, TICKER, "Low")
    if isinstance(high.index, pd.DatetimeIndex) and high.index.tz is not None:
        high = high.copy()
        high.index = high.index.tz_localize(None)
    if isinstance(low.index, pd.DatetimeIndex) and low.index.tz is not None:
        low = low.copy()
        low.index = low.index.tz_localize(None)

    high = high.reindex(price_index)
    low = low.reindex(price_index)
    close = pd.Series(
        pd.Series(close_series).astype(float).to_numpy(),
        index=price_index,
        name=getattr(close_series, "name", None),
    )

    finite_close = np.isfinite(close.to_numpy(dtype=float))
    if not finite_close.any():
        raise ValueError("ADX close_series contains no finite prices.")
    valid_mask = pd.Series(finite_close, index=close.index)
    high_vals = high.loc[valid_mask].to_numpy(dtype=float)
    low_vals = low.loc[valid_mask].to_numpy(dtype=float)
    if (not np.isfinite(high_vals).all()) or (not np.isfinite(low_vals).all()):
        raise ValueError(
            "High/Low data is missing or non-finite for one or more dates in the selected price series."
        )
    if (high_vals < low_vals).any():
        raise ValueError("ADX requires High >= Low on all valid close dates.")

    _adx_ohlc_cache[cache_key] = (high, low, close)
    return _adx_ohlc_cache[cache_key]


def _compute_adx_di_lines(close_series, di_length):
    """TA-Lib +DI/-DI for a DI length; cached so each length is computed once per series."""
    di_length = validate_adx_params(di_length, di_length)[0]
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")
    cache_key = (id(close_series), di_length)
    if cache_key in _adx_di_cache:
        return _adx_di_cache[cache_key]

    high, low, close = get_adx_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= di_length:
        raise ValueError("Not enough price bars to compute ADX DI lines for the selected DI length.")

    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    close_vals = close.to_numpy(dtype=float)
    try:
        plus_di_arr = talib.PLUS_DI(high_vals, low_vals, close_vals, timeperiod=di_length)
        minus_di_arr = talib.MINUS_DI(high_vals, low_vals, close_vals, timeperiod=di_length)
    except Exception as exc:
        raise ValueError(
            f"talib PLUS_DI/MINUS_DI failed for di_length={di_length}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    plus_di = pd.Series(plus_di_arr, index=close.index)
    minus_di = pd.Series(minus_di_arr, index=close.index)
    if (~(plus_di.isna() | minus_di.isna())).sum() == 0:
        raise ValueError("ADX DI lines could not be computed for the selected DI length.")

    _adx_di_cache[cache_key] = (plus_di, minus_di, close)
    return _adx_di_cache[cache_key]


def _wilder_smooth_dx(dx_values, adx_smoothing):
    """Wilder/RMA smooth of DX -> ADX (same DI lines for any DI length / smoothing pair)."""
    dx = np.asarray(dx_values, dtype=float)
    adx = np.full(dx.shape, np.nan, dtype=float)
    if adx_smoothing <= 0:
        raise ValueError("ADX smoothing must be a positive integer.")
    n = int(dx.size)
    if n == 0:
        return adx

    # Seed with SMA of the first contiguous `adx_smoothing` finite DX values,
    # then Wilder-smooth — common ADX / TA-Lib-style initialization.
    run = 0
    seed_end = None
    for i in range(n):
        if np.isfinite(dx[i]):
            run += 1
            if run == adx_smoothing:
                seed_end = i
                break
        else:
            run = 0
    if seed_end is None:
        return adx

    seed_start = seed_end - adx_smoothing + 1
    prev = float(np.mean(dx[seed_start : seed_end + 1]))
    adx[seed_end] = prev
    for i in range(seed_end + 1, n):
        if not np.isfinite(dx[i]):
            adx[i] = np.nan
            continue
        prev = ((prev * (adx_smoothing - 1)) + float(dx[i])) / adx_smoothing
        adx[i] = prev
    return adx


def compute_adx(close_series, di_length, adx_smoothing):
    di_length, adx_smoothing = validate_adx_params(di_length, adx_smoothing)
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")
    cache_key = (id(close_series), di_length, adx_smoothing)
    if cache_key in _adx_series_cache:
        return _adx_series_cache[cache_key]

    plus_di, minus_di, close = _compute_adx_di_lines(close_series, di_length)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    # DI warmup + ADX seed window
    min_bars = di_length + adx_smoothing
    if finite_bars < min_bars:
        raise ValueError(
            "Not enough price bars to compute ADX for the selected parameter combination "
            f"(need at least di_length + adx_smoothing = {min_bars} finite bars)."
        )

    plus_vals = plus_di.to_numpy(dtype=float)
    minus_vals = minus_di.to_numpy(dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        denom = plus_vals + minus_vals
        dx_vals = np.where(
            np.isfinite(denom) & (denom != 0.0),
            100.0 * np.abs(plus_vals - minus_vals) / denom,
            np.nan,
        )
    if not np.isfinite(dx_vals).any():
        raise ValueError("ADX DX could not be computed for the selected parameter combination.")

    try:
        adx_arr = _wilder_smooth_dx(dx_vals, adx_smoothing)
    except Exception as exc:
        raise ValueError(
            f"ADX smoothing failed for di_length={di_length}, "
            f"adx_smoothing={adx_smoothing}: {type(exc).__name__}: {exc}"
        ) from exc

    adx = pd.Series(adx_arr, index=close.index)
    if (~(plus_di.isna() | minus_di.isna() | adx.isna())).sum() == 0:
        raise ValueError("ADX could not be computed for the selected parameter combination.")

    _adx_series_cache[cache_key] = (plus_di, minus_di, adx)
    return _adx_series_cache[cache_key]


def build_adx_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ADX, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("adx_di_length", "adx_smoothing_timeperiod")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ADX params missing keys: {missing_keys}")
    if "ADX_TREND_LEVEL" not in globals():
        raise ValueError("ADX_TREND_LEVEL is missing. Re-run cell 5 (Define Indicator Grids).")
    trend_level = validate_adx_trend_level(ADX_TREND_LEVEL)

    plus_di, minus_di, adx = compute_adx(
        close_series,
        params["adx_di_length"],
        params["adx_smoothing_timeperiod"],
    )
    if len(plus_di) != len(close_series):
        raise ValueError(
            "ADX output length does not match close_series; check OHLC alignment."
        )
    # Require DI lines and ADX valid on this bar and the prior bar so warmup NaNs
    # do not register as false +DI/-DI crosses (same pattern as AROON/KAMA/RSI).
    valid = plus_di.notna() & minus_di.notna() & adx.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: +DI crosses above -DI while ADX confirms a strong uptrend
    entries = (
        plus_di.vbt.crossed_above(minus_di)
        & (adx > trend_level)
        & valid_cross
    ).fillna(False)
    # Sell: -DI crosses above +DI while ADX confirms a strong downtrend
    exits = (
        minus_di.vbt.crossed_above(plus_di)
        & (adx > trend_level)
        & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_adx_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ADX, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("adx_di_length", "adx_smoothing_timeperiod")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ADX params missing keys: {missing_keys}")
    return (
        f"ADX(DI={params['adx_di_length']},"
        f"Smooth={params['adx_smoothing_timeperiod']})"
    )


def validate_adx_sensitivity_params(di_length, adx_smoothing, adx_smoothing_anchor):
    di_length, adx_smoothing = validate_adx_params(di_length, adx_smoothing)
    try:
        anchor = int(adx_smoothing_anchor)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "ADX sensitivity anchor must be an integer matching the ADX smoothing timeperiod."
        ) from exc
    if anchor != adx_smoothing:
        raise ValueError("ADX sensitivity anchor must match the ADX smoothing timeperiod.")
    return di_length, adx_smoothing, adx_smoothing


# DONCHIAN upper-band breakout strategy (TA-Lib MAX/MIN timeperiod; sell on middle-band cross)
def validate_donchian_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"DONCHIAN timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("DONCHIAN timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "DONCHIAN timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("DONCHIAN timeperiod must be a positive integer.")
    return timeperiod


_donchian_series_cache = {}


def get_donchian_ohlc(close_series):
    """Reuse AROON's High/Low alignment cache — same extract, no second OHLC pass."""
    if "get_aroon_ohlc" not in globals():
        raise ValueError(
            "Shared OHLC helper get_aroon_ohlc is missing. "
            "Run cell 5 (Define Indicator Grids) fully (AROON section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("DONCHIAN close_series is empty.")
    try:
        return get_aroon_ohlc(close_series)
    except ValueError as exc:
        msg = str(exc)
        if msg.startswith("AROON "):
            raise ValueError("DONCHIAN " + msg[len("AROON "):]) from exc
        if "AROON requires" in msg:
            raise ValueError(msg.replace("AROON requires", "DONCHIAN requires", 1)) from exc
        raise


def compute_donchian(close_series, timeperiod):
    timeperiod = validate_donchian_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("DONCHIAN close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _donchian_series_cache:
        return _donchian_series_cache[cache_key]

    high, low, close = get_donchian_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= timeperiod:
        raise ValueError("Not enough price bars to compute DONCHIAN for the selected timeperiod.")

    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    try:
        # TA-Lib has no DONCHIAN; upper/lower from MAX/MIN over High/Low.
        upper_arr = talib.MAX(high_vals, timeperiod=timeperiod)
        lower_arr = talib.MIN(low_vals, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib MAX/MIN failed for DONCHIAN timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    upper = pd.Series(upper_arr, index=close.index)
    lower = pd.Series(lower_arr, index=close.index)
    middle = (upper + lower) / 2.0
    if (~(upper.isna() | lower.isna() | middle.isna())).sum() == 0:
        raise ValueError("DONCHIAN could not be computed for the selected parameter combination.")

    _donchian_series_cache[cache_key] = (close, upper, middle, lower)
    return _donchian_series_cache[cache_key]


def build_donchian_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for DONCHIAN, got {type(params).__name__}.")
    missing_keys = [key for key in ("donchian_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"DONCHIAN params missing keys: {missing_keys}")

    close, upper, middle, _lower = compute_donchian(
        close_series, params["donchian_timeperiod"]
    )
    if len(close) != len(close_series):
        raise ValueError(
            "DONCHIAN output length does not match close_series; check OHLC alignment."
        )
    # Use prior bar's channel so close can break above upper (same-bar MAX(High)
    # embeds today's high and makes close > upper impossible).
    prior_upper = upper.shift(1)
    prior_middle = middle.shift(1)
    valid = prior_upper.notna() & prior_middle.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: close crosses above the prior upper band (uptrend breakout)
    entries = (close.vbt.crossed_above(prior_upper) & valid_cross).fillna(False)
    # Sell: close crosses below the prior middle band (trailing stop)
    exits = (close.vbt.crossed_below(prior_middle) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX pattern). close.index may be
    # tz-stripped inside get_aroon_ohlc and would mis-align the backtest.
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_donchian_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for DONCHIAN, got {type(params).__name__}.")
    missing_keys = [key for key in ("donchian_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"DONCHIAN params missing keys: {missing_keys}")
    return f"DONCHIAN({params['donchian_timeperiod']})"


def validate_donchian_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_donchian_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "DONCHIAN sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("DONCHIAN sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# TRIX zero-line crossover strategy (TA-Lib TRIX timeperiod; buy/sell on cross of 0)
def validate_trix_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"TRIX timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("TRIX timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "TRIX timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("TRIX timeperiod must be a positive integer.")
    return timeperiod


_trix_series_cache = {}


def compute_trix(close_series, timeperiod):
    timeperiod = validate_trix_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("TRIX close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _trix_series_cache:
        return _trix_series_cache[cache_key]

    close = pd.Series(close_series, index=close_series.index).astype(float)
    close_values = close.to_numpy(dtype=float)
    if not np.isfinite(close_values).any():
        raise ValueError("TRIX close_series contains no finite prices.")

    # TRIX is ROC of a triple EMA; needs far more warmup than timeperiod+1.
    # TA-Lib TRIX lookback is 3*(timeperiod-1)+1; need one more bar for a usable value.
    min_bars = 3 * (timeperiod - 1) + 2
    if np.isfinite(close_values).sum() < min_bars:
        raise ValueError(
            "Not enough price bars to compute TRIX for the selected timeperiod "
            f"(need >= {min_bars} finite bars for timeperiod={timeperiod})."
        )

    try:
        trix_arr = talib.TRIX(close_values, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib.TRIX failed for timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    trix = pd.Series(trix_arr, index=close.index)
    if trix.notna().sum() == 0:
        raise ValueError("TRIX could not be computed for the selected parameter combination.")
    _trix_series_cache[cache_key] = trix
    return trix


def build_trix_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for TRIX, got {type(params).__name__}.")
    missing_keys = [key for key in ("trix_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"TRIX params missing keys: {missing_keys}")
    if "TRIX_CROSS_LEVEL" not in globals():
        raise ValueError("TRIX_CROSS_LEVEL is missing. Re-run cell 5 (Define Indicator Grids).")

    trix = compute_trix(close_series, params["trix_timeperiod"])
    valid = trix.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: TRIX crosses above 0 (bullish momentum)
    entries = (
        (trix > TRIX_CROSS_LEVEL) & (trix.shift(1) <= TRIX_CROSS_LEVEL) & valid_cross
    ).fillna(False)
    # Sell: TRIX crosses below 0 (bearish momentum)
    exits = (
        (trix < TRIX_CROSS_LEVEL) & (trix.shift(1) >= TRIX_CROSS_LEVEL) & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_trix_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for TRIX, got {type(params).__name__}.")
    missing_keys = [key for key in ("trix_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"TRIX params missing keys: {missing_keys}")
    return f"TRIX({params['trix_timeperiod']})"


def validate_trix_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_trix_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "TRIX sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("TRIX sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# VORTEX VI+/VI- crossover strategy (TA-Lib has no VORTEX; TRANGE/SUM over timeperiod)
def validate_vortex_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"VORTEX timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("VORTEX timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "VORTEX timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("VORTEX timeperiod must be a positive integer.")
    return timeperiod


_vortex_series_cache = {}
_vortex_base_cache = {}


def get_vortex_ohlc(close_series):
    """Reuse AROON's High/Low alignment cache — same extract, no second OHLC pass."""
    if "get_aroon_ohlc" not in globals():
        raise ValueError(
            "Shared OHLC helper get_aroon_ohlc is missing. "
            "Run cell 5 (Define Indicator Grids) fully (AROON section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    try:
        return get_aroon_ohlc(close_series)
    except ValueError as exc:
        msg = str(exc)
        if msg.startswith("AROON "):
            raise ValueError("VORTEX " + msg[len("AROON "):]) from exc
        if "AROON requires" in msg:
            raise ValueError(msg.replace("AROON requires", "VORTEX requires", 1)) from exc
        raise


def _get_vortex_base_arrays(close_series):
    """Cache TRANGE and VM+/- once per price series (independent of timeperiod)."""
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    cache_key = id(close_series)
    if cache_key in _vortex_base_cache:
        return _vortex_base_cache[cache_key]

    high, low, close = get_vortex_ohlc(close_series)
    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    close_vals = close.to_numpy(dtype=float)
    if not (len(high_vals) == len(low_vals) == len(close_vals)):
        raise ValueError("VORTEX High/Low/Close lengths do not match after OHLC alignment.")

    try:
        # TA-Lib has no VORTEX; VI+/VI- from vortex movement vs true range.
        tr_arr = talib.TRANGE(high_vals, low_vals, close_vals)
    except Exception as exc:
        raise ValueError(
            f"talib.TRANGE failed for VORTEX: {type(exc).__name__}: {exc}"
        ) from exc

    # First bar has no prior High/Low; leave VM undefined (NaN), not 0.
    vm_plus = np.full(len(high_vals), np.nan, dtype=float)
    vm_minus = np.full(len(high_vals), np.nan, dtype=float)
    vm_plus[1:] = np.abs(high_vals[1:] - low_vals[:-1])
    vm_minus[1:] = np.abs(low_vals[1:] - high_vals[:-1])

    _vortex_base_cache[cache_key] = (tr_arr, vm_plus, vm_minus, close)
    return _vortex_base_cache[cache_key]


def compute_vortex(close_series, timeperiod):
    timeperiod = validate_vortex_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _vortex_series_cache:
        return _vortex_series_cache[cache_key]

    tr_arr, vm_plus, vm_minus, close = _get_vortex_base_arrays(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    # First VI needs a prior bar for VM+/- and `timeperiod` finite TR/VM values.
    min_bars = timeperiod + 1
    if finite_bars < min_bars:
        raise ValueError(
            "Not enough price bars to compute VORTEX for the selected timeperiod "
            f"(need >= {min_bars} finite bars for timeperiod={timeperiod})."
        )

    try:
        sum_vm_plus = talib.SUM(vm_plus, timeperiod=timeperiod)
        sum_vm_minus = talib.SUM(vm_minus, timeperiod=timeperiod)
        sum_tr = talib.SUM(tr_arr, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib.SUM failed for VORTEX timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    vi_plus_arr = np.full(len(sum_tr), np.nan, dtype=float)
    vi_minus_arr = np.full(len(sum_tr), np.nan, dtype=float)
    valid_tr = np.isfinite(sum_tr) & (sum_tr != 0)
    # Only divide where the TR sum is finite and non-zero.
    vi_plus_arr[valid_tr] = sum_vm_plus[valid_tr] / sum_tr[valid_tr]
    vi_minus_arr[valid_tr] = sum_vm_minus[valid_tr] / sum_tr[valid_tr]

    vi_plus = pd.Series(vi_plus_arr, index=close.index)
    vi_minus = pd.Series(vi_minus_arr, index=close.index)
    if (~(vi_plus.isna() | vi_minus.isna())).sum() == 0:
        raise ValueError("VORTEX could not be computed for the selected parameter combination.")

    _vortex_series_cache[cache_key] = (vi_plus, vi_minus)
    return _vortex_series_cache[cache_key]


def build_vortex_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for VORTEX, got {type(params).__name__}.")
    missing_keys = [key for key in ("vortex_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"VORTEX params missing keys: {missing_keys}")

    vi_plus, vi_minus = compute_vortex(close_series, params["vortex_timeperiod"])
    if len(vi_plus) != len(close_series) or len(vi_minus) != len(close_series):
        raise ValueError(
            "VORTEX output length does not match close_series; check OHLC alignment."
        )
    # Require both lines valid on this bar and the prior bar so warmup NaNs
    # do not register as false VI+/VI- crosses (same pattern as AROON/KAMA/RSI).
    valid = vi_plus.notna() & vi_minus.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: VI+ crosses upward from below VI-
    entries = (vi_plus.vbt.crossed_above(vi_minus) & valid_cross).fillna(False)
    # Sell: VI- crosses upward from below VI+
    exits = (vi_minus.vbt.crossed_above(vi_plus) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )

def format_vortex_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for VORTEX, got {type(params).__name__}.")
    missing_keys = [key for key in ("vortex_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"VORTEX params missing keys: {missing_keys}")
    return f"VORTEX({params['vortex_timeperiod']})"


def validate_vortex_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_vortex_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "VORTEX sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("VORTEX sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# ALMA crossover strategy (custom ALMA: fast/slow length + offset hundredths + sigma)
if (
    "validate_alma_params" not in globals()
    or "_compute_alma_core" not in globals()
):
    raise ValueError("ALMA helpers are missing. Run cell 3 (TA-Lib indicators) first.")


_alma_series_cache = {}


def validate_alma_crossover_params(fast_length, slow_length, offset, sigma):
    fast_length, offset, sigma = validate_alma_params(fast_length, offset, sigma)
    slow_length, offset, sigma = validate_alma_params(slow_length, offset, sigma)
    if fast_length >= slow_length:
        raise ValueError("ALMA fast length must be less than slow length.")
    return fast_length, slow_length, offset, sigma


def get_alma_series(close_series, length, offset, sigma):
    length, offset, sigma = validate_alma_params(length, offset, sigma)
    if close_series is None or len(close_series) == 0:
        raise ValueError("ALMA close_series is empty.")
    if not hasattr(close_series, "index"):
        raise TypeError(
            f"ALMA close_series must be a pandas Series; got {type(close_series).__name__}."
        )
    cache_key = (id(close_series), length, offset, sigma)
    if cache_key not in _alma_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        close_values = close.to_numpy(dtype=float)
        if not np.isfinite(close_values).any():
            raise ValueError("ALMA close_series contains no finite prices.")
        try:
            alma_line = pd.Series(
                _compute_alma_core(close_values, length, offset, sigma),
                index=close.index,
            )
        except ValueError as exc:
            raise ValueError(
                f"ALMA could not be computed for length={length}, offset={offset}, "
                f"sigma={sigma}: {exc}"
            ) from exc
        if alma_line.notna().sum() == 0:
            raise ValueError("ALMA could not be computed for the selected parameter combination.")
        _alma_series_cache[cache_key] = (close, alma_line)
    return _alma_series_cache[cache_key]


def build_alma_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ALMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("alma_fast_length", "alma_slow_length", "alma_offset", "alma_sigma")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ALMA params missing keys: {missing_keys}")

    fast_length, slow_length, offset, sigma = validate_alma_crossover_params(
        params["alma_fast_length"],
        params["alma_slow_length"],
        params["alma_offset"],
        params["alma_sigma"],
    )

    close, alma_fast = get_alma_series(close_series, fast_length, offset, sigma)
    _, alma_slow = get_alma_series(close_series, slow_length, offset, sigma)
    if len(alma_fast) != len(close_series) or len(alma_slow) != len(close_series):
        raise ValueError(
            "ALMA output length does not match close_series; check close alignment."
        )
    # Require both ALMAs and close valid on this bar and the prior bar so warmup NaNs
    # do not register as false fast/slow crosses (same pattern as AROON/KAMA/RSI).
    valid = alma_fast.notna() & alma_slow.notna() & close.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: fast ALMA crosses above slow ALMA
    entries = (alma_fast.vbt.crossed_above(alma_slow) & valid_cross).fillna(False)
    # Sell: fast ALMA crosses below slow ALMA
    exits = (alma_fast.vbt.crossed_below(alma_slow) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_alma_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ALMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("alma_fast_length", "alma_slow_length", "alma_offset", "alma_sigma")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ALMA params missing keys: {missing_keys}")
    fast_length, slow_length, offset, sigma = validate_alma_crossover_params(
        params["alma_fast_length"],
        params["alma_slow_length"],
        params["alma_offset"],
        params["alma_sigma"],
    )
    return f"ALMA({fast_length},{slow_length},{offset / 100:.2f},{sigma})"


# LIVE TRADING: parameter grids are not searched here.
# Ensemble params come from master_leaderboard_ensembles.json instead.
ema_combinations = []
macd_combinations = []
aroon_combinations = []
stc_combinations = []
kama_combinations = []
supertrend_combinations = []
kalman_combinations = []
rsi_combinations = []
adx_combinations = []
donchian_combinations = []
trix_combinations = []
vortex_combinations = []
alma_combinations = []

indicator_grids = {
    "Triple EMA": {
        "description": "Triple EMA crossover strategy",
        "params": ema_combinations,
        "build_signals": build_triple_ema_signals,
        "format_params": format_triple_ema_params
    },
    "MACD": {
        "description": "MACD line crossing its signal line",
        "params": macd_combinations,
        "build_signals": build_macd_signals,
        "format_params": format_macd_params
    },
    "AROON": {
        "description": "AROON Up/Down crossover: buy when Up crosses above Down and Up > entry level, sell when Down crosses above Up",
        "params": aroon_combinations,
        "build_signals": build_aroon_signals,
        "format_params": format_aroon_params
    },
    "STC": {
        "description": "Schaff Trend Cycle: buy on cross above 50, sell on cross below 50",
        "params": stc_combinations,
        "build_signals": build_stc_signals,
        "format_params": format_stc_params
    },
    "KAMA": {
        "description": "Price crossing KAMA line: buy on cross above, sell on cross below",
        "params": kama_combinations,
        "build_signals": build_kama_signals,
        "format_params": format_kama_params
    },
    "SUPERTREND": {
        "description": "SUPERTREND flip: buy when close above red line, sell when close below green line",
        "params": supertrend_combinations,
        "build_signals": build_supertrend_signals,
        "format_params": format_supertrend_params
    },
    "KALMAN": {
        "description": "Price crossing Kalman filter: buy on cross above, sell on cross below",
        "params": kalman_combinations,
        "build_signals": build_kalman_signals,
        "format_params": format_kalman_params
    },
    "RSI": {
        "description": "RSI crossing 50: buy on cross above, sell on cross below",
        "params": rsi_combinations,
        "build_signals": build_rsi_signals,
        "format_params": format_rsi_params
    },
    "ADX": {
        "description": "ADX +DI/-DI crossover: buy when +DI crosses above -DI with ADX > trend level, sell when -DI crosses above +DI with ADX > trend level",
        "params": adx_combinations,
        "build_signals": build_adx_signals,
        "format_params": format_adx_params
    },
    "DONCHIAN": {
        "description": "DONCHIAN breakout: buy when close crosses above prior upper band, sell when close crosses below prior middle band",
        "params": donchian_combinations,
        "build_signals": build_donchian_signals,
        "format_params": format_donchian_params
    },
    "TRIX": {
        "description": "TRIX crossing 0: buy on cross above, sell on cross below",
        "params": trix_combinations,
        "build_signals": build_trix_signals,
        "format_params": format_trix_params
    },
    "VORTEX": {
        "description": "VORTEX VI+/VI- crossover: buy when VI+ crosses above VI-, sell when VI- crosses above VI+",
        "params": vortex_combinations,
        "build_signals": build_vortex_signals,
        "format_params": format_vortex_params
    },
    "ALMA": {
        "description": "ALMA fast/slow crossover: buy when fast crosses above slow, sell when fast crosses below slow",
        "params": alma_combinations,
        "build_signals": build_alma_signals,
        "format_params": format_alma_params
    },
}

active_indicator_names = ["Triple EMA", "MACD", "AROON", "STC", "KAMA", "SUPERTREND", "KALMAN", "RSI", "ADX", "DONCHIAN", "TRIX", "VORTEX", "ALMA"]
missing_indicator_names = [
    name for name in active_indicator_names
    if name not in indicator_grids
]
if missing_indicator_names:
    raise KeyError(f"Active indicators are not defined: {missing_indicator_names}")

active_indicator_grids = {
    name: indicator_grids[name]
    for name in active_indicator_names
}


def get_active_indicator_spec(indicator_name):
    if indicator_name not in active_indicator_grids:
        available = ", ".join(active_indicator_grids.keys())
        raise KeyError(f"Unknown indicator '{indicator_name}'. Available indicators: {available}")
    return active_indicator_grids[indicator_name]


def require_params_dict(params, indicator_name):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for {indicator_name}, got {type(params).__name__}.")
    return params




INIT_CASH = 100_000
FEE_RATE = 0.0005
SLIPPAGE_RATE = 0.0005


def trade_stats_from_portfolio(pf):
    """Extract trade-level stats; safe when there are zero trades."""
    trades = pf.trades
    total_trades = len(trades)
    stats = {
        "total_trades": total_trades,
        "win_rate_pct": np.nan,
        "profit_factor": np.nan,
        "expectancy": np.nan,
        "avg_win": np.nan,
        "avg_loss": np.nan,
    }
    if total_trades <= 0:
        return stats

    tr = trades.returns.values if hasattr(trades.returns, "values") else np.asarray(trades.returns)
    if tr.size == 0:
        return stats

    pos = tr[tr > 0]
    neg = tr[tr < 0]
    stats["win_rate_pct"] = (len(pos) / len(tr)) * 100.0
    gains = float(pos.sum()) if len(pos) else 0.0
    losses = float(abs(neg.sum())) if len(neg) else 0.0
    stats["profit_factor"] = (gains / losses) if losses > 0 else (np.inf if gains > 0 else np.nan)
    stats["expectancy"] = float(tr.mean())
    stats["avg_win"] = float(pos.mean()) if len(pos) else np.nan
    stats["avg_loss"] = float(abs(neg.mean())) if len(neg) else np.nan
    return stats


def run_signal_backtest(price, entries, exits):
    """Shared Portfolio.from_signals wrapper used by the ensemble cell (cell 7)."""
    close_arg = price.to_numpy(dtype=float) if hasattr(price, "to_numpy") else np.asarray(price, dtype=float)
    if hasattr(entries, "to_numpy"):
        entries = entries.to_numpy(dtype=bool)
    if hasattr(exits, "to_numpy"):
        exits = exits.to_numpy(dtype=bool)
    return vbt.Portfolio.from_signals(
        close=close_arg,
        entries=entries,
        exits=exits,
        init_cash=INIT_CASH,
        fees=FEE_RATE,
        slippage=SLIPPAGE_RATE,
        freq=FREQ,
    )


def fmt_metric(value, spec=".3f", suffix=""):
    """Format metrics without crashing on NaN/None/inf."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "n/a"
    try:
        return format(float(value), spec) + suffix
    except (TypeError, ValueError):
        return "n/a"


def safe_float(value, default=np.nan):
    """Convert to float; return default for missing/invalid/non-finite values."""
    try:
        out = float(value)
        return out if np.isfinite(out) else default
    except (TypeError, ValueError):
        return default


_tied_is_sharpe_notes_printed = set()


def displayed_is_top_n(indicator_results, top_n=5):
    """Top-N rows shown in the in-sample results table."""
    if "sharpe_ratio" not in indicator_results.columns:
        raise KeyError("indicator_results is missing required column: sharpe_ratio")
    return indicator_results.dropna(subset=["sharpe_ratio"]).nlargest(top_n, "sharpe_ratio")


def is_sharpe_tied_at_best(indicator_results, rtol=1e-9, atol=1e-12):
    valid = indicator_results.dropna(subset=["sharpe_ratio"])
    if valid.empty:
        return False
    best_sharpe = valid["sharpe_ratio"].max()
    tied_count = np.isclose(valid["sharpe_ratio"], best_sharpe, rtol=rtol, atol=atol).sum()
    return int(tied_count) > 1


def maybe_print_tied_is_sharpe_note(indicator_name, indicator_results, used_param_displays, context, top_n=5):
    if indicator_name in _tied_is_sharpe_notes_printed:
        return

    required_columns = {"sharpe_ratio", "param_display"}
    missing_columns = required_columns.difference(indicator_results.columns)
    if missing_columns:
        return

    used_labels = {
        str(label).strip()
        for label in used_param_displays
        if label is not None and pd.notna(label) and str(label).strip() and str(label).strip().lower() != "nan"
    }
    if not used_labels:
        return

    display_top = displayed_is_top_n(indicator_results, top_n)
    if display_top.empty:
        return

    display_labels = {
        str(label).strip()
        for label in display_top["param_display"].tolist()
        if label is not None and pd.notna(label) and str(label).strip() and str(label).strip().lower() != "nan"
    }
    outside_display = sorted(used_labels - display_labels)
    if not outside_display or not is_sharpe_tied_at_best(indicator_results):
        return

    best_sharpe = float(display_top["sharpe_ratio"].iloc[0])
    print(
        f"Note ({context}): {indicator_name} has multiple parameter sets tied at the "
        f"best in-sample Sharpe ({best_sharpe:.3f}). The in-sample top-{top_n} table shows one "
        f"valid tied set; this step used different tied equivalent(s): "
        f"{', '.join(outside_display)}"
    )
    _tied_is_sharpe_notes_printed.add(indicator_name)


def best_is_row(indicator_results):
    """Pick the single best in-sample row by Sharpe (matches the IS results table)."""
    ranked = displayed_is_top_n(indicator_results, 1)
    if ranked.empty:
        return None
    return ranked.iloc[0]


def best_oos_row(oos_results_df):
    """Pick the single best OOS row by Sharpe among validated IS top-N candidates."""
    valid = oos_results_df.dropna(subset=["OOS_Sharpe"])
    if valid.empty:
        return None
    return valid.loc[valid["OOS_Sharpe"].idxmax()]


total_grid_combinations = sum(len(spec["params"]) for spec in active_indicator_grids.values())

print("Active indicator signal builders ready for live trading:")
for indicator_name, spec in active_indicator_grids.items():
    print(f"  - {indicator_name}: {spec['description']}")
print(
    f"Grid search disabled (params={total_grid_combinations}). "
    "Live path uses leaderboard param_display values only."
)


Active indicator signal builders ready for live trading:
  - Triple EMA: Triple EMA crossover strategy
  - MACD: MACD line crossing its signal line
  - AROON: AROON Up/Down crossover: buy when Up crosses above Down and Up > entry level, sell when Down crosses above Up
  - STC: Schaff Trend Cycle: buy on cross above 50, sell on cross below 50
  - KAMA: Price crossing KAMA line: buy on cross above, sell on cross below
  - SUPERTREND: SUPERTREND flip: buy when close above red line, sell when close below green line
  - KALMAN: Price crossing Kalman filter: buy on cross above, sell on cross below
  - RSI: RSI crossing 50: buy on cross above, sell on cross below
  - ADX: ADX +DI/-DI crossover: buy when +DI crosses above -DI with ADX > trend level, sell when -DI crosses above +DI with ADX > trend level
  - DONCHIAN: DONCHIAN breakout: buy when close crosses above prior upper band, sell when close crosses below prior middle band
  - TRIX: TRIX crossing 0: buy on cross above, sell on cross belo

DATA PREPARATION - PER-TICKER RETURNS AND POSITION FILES
--------------------------------------------------------

This section writes one pre-processed file per master-leaderboard ticker, then aligns those files for live signals.

- Best ensembles are loaded from `Ensemble_results/master_leaderboard_ensembles.json` (no re-run of the ensemble grid).
- For each ticker, member signals are rebuilt from `members` + `param_display` on that ticker's full Close series and combined with the saved `combine_rule` (OR).
- The position is tagged at that session's close (no extra 1-session shift). Allocation then applies it at the start of the ticker's next native session (europe / na / crypto).
- Each file contains `Date`, a Buy & Hold return column (`BnH`), a Strategy return column (`Strategy`), and a Position column (`Position`).
- Files are written to `RAAM_data/{ticker_slug}_raam.csv`.
- Live alignment uses an outer join on equity session dates (majority of non-crypto tickers printing). Holiday gaps are filled with 0% BnH and forward-filled Positions so one missing market (e.g. SMH.L on 2026-09-14) does not freeze the signal date.

---

In [6]:
# DATA PREPARATION: one file per master-leaderboard ticker
# Columns: Date, {ticker}_BnH, {ticker}_Strategy, {ticker}_Position

import ast
import os
import re

RAAM_DATA_DIR = "RAAM_data"
RAAM_FILE_SUFFIX = "_raam.csv"
ENSEMBLE_LEADERBOARD_RESULTS_DIR = "Ensemble_results"
ENSEMBLE_LEADERBOARD_RESULTS_FILENAME = "master_leaderboard_ensembles.json"
RAAM_MIN_MEMBERS = 2
_RAAM_WINDOWS_RESERVED_NAMES = {
    "con", "prn", "aux", "nul",
    "com1", "com2", "com3", "com4", "com5", "com6", "com7", "com8", "com9",
    "lpt1", "lpt2", "lpt3", "lpt4", "lpt5", "lpt6", "lpt7", "lpt8", "lpt9",
}

# Local copy of param keys so this cell does not need the ensemble grid-search cell.
_RAAM_PARAM_KEYS = {
    "Triple EMA": ("ema1_period", "ema2_period", "ema3_period"),
    "MACD": ("macd_fast_period", "macd_slow_period", "macd_signal_period"),
    "AROON": ("aroon_timeperiod",),
    "STC": ("stc_ema_short_period", "stc_ema_long_period", "stc_cycle_period"),
    "KAMA": ("kama_period", "kama_fast_ema_constant", "kama_slow_ema_constant"),
    "SUPERTREND": ("supertrend_atr_length", "supertrend_factor"),
    "KALMAN": ("kalman_process_noise", "kalman_measurement_noise"),
    "RSI": ("rsi_timeperiod",),
    "ADX": ("adx_di_length", "adx_smoothing_timeperiod"),
    "DONCHIAN": ("donchian_timeperiod",),
    "TRIX": ("trix_timeperiod",),
    "VORTEX": ("vortex_timeperiod",),
    "ALMA": ("alma_fast_length", "alma_slow_length", "alma_offset", "alma_sigma"),
}

# Named CSV forms: SUPERTREND(ATR=36, Factor=4), KALMAN(Q=1, R=113), ADX(DI=19, Smooth=44)
_RAAM_NAMED_ALIASES = {
    "SUPERTREND": ("atr", "factor"),
    "KALMAN": ("q", "r"),
    "ADX": ("di", "smooth"),
}

_RAAM_TICKER_ERRORS = (
    ValueError,
    KeyError,
    TypeError,
    OSError,
    AttributeError,
    IndexError,
    RuntimeError,
)

required_helpers = [
    "close_by_ticker",
    "get_active_indicator_spec",
    "run_signal_backtest",
    "safe_float",
]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise NameError(
        "Missing required helpers/data from earlier cells: "
        + ", ".join(missing_helpers)
    )
if not isinstance(close_by_ticker, dict) or not close_by_ticker:
    raise ValueError("close_by_ticker is missing or empty. Run the price-prep cell first.")


def _raam_notebook_dir():
    """Best-effort project/notebook directory (cwd fallback in Jupyter)."""
    try:
        return os.path.dirname(os.path.abspath(__file__))
    except NameError:
        return os.getcwd()


def _raam_leaf(name, label, require_csv=False, require_json=False):
    """Require a single path component (optional .csv / .json suffix)."""
    if require_csv and require_json:
        raise ValueError(f"{label}: require_csv and require_json are mutually exclusive")
    text = str(name).strip() if name is not None else ""
    if not text:
        raise ValueError(f"{label} must be a non-empty name; got {name!r}")
    if os.path.isabs(text) or os.path.dirname(text) not in ("", "."):
        raise ValueError(f"{label} must be a single path component; got {name!r}")
    if text in {".", ".."} or os.sep in text or (os.altsep and os.altsep in text):
        raise ValueError(f"{label} must be a single path component; got {name!r}")
    need_suffix = require_csv or require_json
    stem = text.lower().rsplit(".", 1)[0] if need_suffix else text.lower()
    if stem in _RAAM_WINDOWS_RESERVED_NAMES:
        raise ValueError(f"{label} uses a reserved name; got {name!r}")
    if require_csv and not text.lower().endswith(".csv"):
        raise ValueError(f"{label} must end with .csv; got {name!r}")
    if require_json and not text.lower().endswith(".json"):
        raise ValueError(f"{label} must end with .json; got {name!r}")
    return text


def _raam_ticker_slug(ticker):
    """Match indicator-sweep slug rules for `{slug}_raam.csv`."""
    text = str(ticker).strip()
    if not text:
        raise ValueError(f"Could not build results filename from TICKER={ticker!r}")
    slug = re.sub(r"[^a-z0-9]+", "_", text.lower())
    slug = re.sub(r"_+", "_", slug).strip("_")
    if not slug:
        raise ValueError(f"Could not build results filename from TICKER={ticker!r}")
    if slug in _RAAM_WINDOWS_RESERVED_NAMES:
        slug = f"asset_{slug}"
    return slug


def _raam_data_dir():
    """Return RAAM_data next to the notebook, creating it if needed."""
    dir_name = _raam_leaf(RAAM_DATA_DIR, "RAAM_DATA_DIR")
    results_dir = os.path.join(os.path.abspath(_raam_notebook_dir()), dir_name)
    if os.path.exists(results_dir) and not os.path.isdir(results_dir):
        raise ValueError(f"RAAM data path exists but is not a directory: {results_dir!r}")
    try:
        os.makedirs(results_dir, exist_ok=True)
    except OSError as exc:
        raise OSError(f"Could not create RAAM data directory {results_dir!r}: {exc}") from exc
    if not os.path.isdir(results_dir):
        raise ValueError(f"RAAM data path exists but is not a directory: {results_dir!r}")
    return results_dir


def _raam_ticker_filepath(ticker, results_dir):
    """Build `{slug}_raam.csv` under an already-resolved RAAM_data directory."""
    filename = _raam_leaf(
        f"{_raam_ticker_slug(ticker)}{RAAM_FILE_SUFFIX}",
        "RAAM filename",
        require_csv=True,
    )
    results_dir = os.path.abspath(results_dir)
    filepath = os.path.abspath(os.path.join(results_dir, filename))
    if os.path.dirname(filepath) != results_dir:
        raise ValueError(f"Refusing RAAM filepath outside {RAAM_DATA_DIR}: {filepath!r}")
    if os.path.exists(filepath) and not os.path.isfile(filepath):
        raise ValueError(f"RAAM path exists but is not a file: {filepath!r}")
    return filepath


def _raam_leaderboard_search_dirs():
    dir_name = _raam_leaf(ENSEMBLE_LEADERBOARD_RESULTS_DIR, "ENSEMBLE_LEADERBOARD_RESULTS_DIR")
    seen = []
    out = []
    for base in (_raam_notebook_dir(), os.getcwd()):
        candidate = os.path.abspath(os.path.join(base, dir_name))
        if candidate in seen:
            continue
        seen.append(candidate)
        out.append(candidate)
    return out


def _raam_leaderboard_filepath():
    """Locate master_leaderboard_ensembles.json (must already exist)."""
    filename = _raam_leaf(
        ENSEMBLE_LEADERBOARD_RESULTS_FILENAME,
        "ENSEMBLE_LEADERBOARD_RESULTS_FILENAME",
        require_json=True,
    )
    searched = []
    for directory in _raam_leaderboard_search_dirs():
        filepath = os.path.abspath(os.path.join(directory, filename))
        searched.append(filepath)
        if os.path.isfile(filepath):
            return filepath
        if os.path.exists(filepath) and not os.path.isfile(filepath):
            raise ValueError(f"Leaderboard path exists but is not a file: {filepath!r}")
    raise FileNotFoundError(
        "Could not resolve master leaderboard JSON. "
        f"Expected filename {filename!r}. Looked in: {searched!r}"
    )


def _raam_require_label(value, name):
    """Return a stripped non-empty label, rejecting missing/NaN placeholders."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        raise ValueError(f"{name} is missing")
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "nat"}:
        raise ValueError(f"{name} is missing or empty; got {value!r}")
    return text


def _raam_members_cell(value):
    """Coerce a leaderboard members cell to a list of indicator names."""
    if isinstance(value, np.ndarray):
        value = value.tolist()
    elif isinstance(value, tuple):
        value = list(value)
    elif isinstance(value, str):
        text = value.strip()
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
            except (SyntaxError, ValueError, TypeError) as exc:
                raise TypeError(
                    f"members string did not parse to a list; got {value!r}"
                ) from exc
            if not isinstance(parsed, (list, tuple)):
                raise TypeError(
                    f"members string did not parse to a list; got {type(parsed).__name__}"
                )
            value = list(parsed)
        else:
            value = [part.strip() for part in text.split(",") if part.strip()]
    elif not isinstance(value, list):
        raise TypeError(
            "members must be a list, tuple, or ndarray of indicator names; "
            f"got {type(value).__name__}"
        )
    members = []
    for item in value:
        if isinstance(item, (list, tuple, dict, np.ndarray)):
            raise TypeError(
                f"members item must be an indicator name, not {type(item).__name__}"
            )
        members.append(_raam_require_label(item, "members item"))
    if len(members) < RAAM_MIN_MEMBERS:
        raise ValueError(
            f"members must have at least {RAAM_MIN_MEMBERS} indicator(s); got {members!r}"
        )
    if len(set(members)) != len(members):
        raise ValueError(f"members contains duplicates: {members!r}")
    return members


def _raam_parse_param_tokens(param_range):
    """Extract the inside of Name(...) from a param_display / param_range label."""
    text = str(param_range).strip()
    match = re.search(r"\((.*)\)\s*$", text)
    if not match:
        raise ValueError(f"Could not parse param_range label: {param_range!r}")
    inside = match.group(1).strip()
    if not inside:
        raise ValueError(f"Empty parameter list in param_range label: {param_range!r}")
    return [token.strip() for token in inside.split(",")]


def _raam_parse_params_from_full_param_range(indicator_name, param_range):
    """Rebuild a params dict from a param_display segment."""
    keys = _RAAM_PARAM_KEYS.get(indicator_name)
    if not keys:
        raise KeyError(f"No param keys registered for indicator {indicator_name!r}")

    tokens = _raam_parse_param_tokens(param_range)
    named = {}
    positional = []
    for token in tokens:
        if "=" in token:
            name, raw = token.split("=", 1)
            named[name.strip().lower()] = raw.strip()
        else:
            positional.append(token)

    if named and positional:
        raise ValueError(
            f"{indicator_name}: mixed named/positional params in {param_range!r}"
        )

    if named:
        aliases = _RAAM_NAMED_ALIASES.get(indicator_name)
        if aliases is None or len(aliases) != len(keys):
            raise ValueError(
                f"{indicator_name}: unsupported named param_range format: {param_range!r}"
            )
        values = []
        for alias in aliases:
            if alias not in named:
                raise KeyError(
                    f"{indicator_name}: missing '{alias}=' in param_range {param_range!r}"
                )
            values.append(named[alias])
    else:
        if len(positional) != len(keys):
            raise ValueError(
                f"{indicator_name}: expected {len(keys)} values in {param_range!r}, "
                f"got {len(positional)}"
            )
        values = positional

    params = {}
    for key, raw in zip(keys, values):
        try:
            number = float(raw)
        except (TypeError, ValueError) as exc:
            raise ValueError(
                f"{indicator_name}: non-numeric param {key}={raw!r} in {param_range!r}"
            ) from exc
        if not np.isfinite(number):
            raise ValueError(
                f"{indicator_name}: non-finite param {key}={raw!r} in {param_range!r}"
            )
        if key == "alma_offset":
            # CSV displays fractional offset (0.14); signal builder stores hundredths (14).
            if abs(number) <= 1.0:
                params[key] = int(round(number * 100.0))
            else:
                params[key] = int(round(number))
        elif number == int(number):
            params[key] = int(number)
        else:
            params[key] = float(number)
    return params


def _raam_require_indicator_param_keys(indicator_name, params):
    """Ensure parsed params include the registered keys for this indicator."""
    keys = _RAAM_PARAM_KEYS.get(indicator_name)
    if not keys:
        raise KeyError(f"No param keys registered for indicator {indicator_name!r}")
    if not isinstance(params, dict):
        raise TypeError(
            f"{indicator_name}: expected params dict, got {type(params).__name__}"
        )
    missing = [key for key in keys if key not in params]
    if missing:
        raise KeyError(f"{indicator_name}: params missing required keys: {missing}")
    return params


def _raam_member_params_from_param_display(members, param_display):
    """Map each ensemble member to params parsed from `NAME NAME(...) | ...`."""
    parts = [p.strip() for p in str(param_display).split("|") if str(p).strip()]
    if len(parts) != len(members):
        raise ValueError(
            f"param_display has {len(parts)} segment(s) but members has {len(members)}: "
            f"param_display={param_display!r}, members={members!r}"
        )
    out = {}
    for member, part in zip(members, parts):
        params = _raam_parse_params_from_full_param_range(member, part)
        out[member] = _raam_require_indicator_param_keys(member, params)
    return out


def _raam_as_bool1d(signal, label):
    """Coerce a signal to a 1-D bool ndarray."""
    arr = np.asarray(signal, dtype=bool)
    if arr.ndim != 1:
        raise ValueError(f"{label} must be 1-D; got shape {arr.shape}")
    return arr


def _raam_or_combine_member_signals(signal_map, members):
    """Logical OR. Any member entry/exit fires."""
    members = list(members)
    if not members:
        raise ValueError("OR combine needs >= 1 member")
    missing = [name for name in members if name not in signal_map]
    if missing:
        raise KeyError(f"missing member signals for: {missing}")

    first_entries, first_exits = signal_map[members[0]]
    first_entries = _raam_as_bool1d(first_entries, f"{members[0]} entries")
    first_exits = _raam_as_bool1d(first_exits, f"{members[0]} exits")
    if first_entries.size == 0 or first_exits.size == 0:
        raise ValueError(f"{members[0]} signals must be non-empty")
    if first_entries.size != first_exits.size:
        raise ValueError(
            f"{members[0]} entries/exits length mismatch: "
            f"{first_entries.size} vs {first_exits.size}"
        )

    n = int(first_entries.size)
    entries_or = first_entries.copy()
    exits_or = first_exits.copy()
    for name in members[1:]:
        entries_i, exits_i = signal_map[name]
        entries_i = _raam_as_bool1d(entries_i, f"{name} entries")
        exits_i = _raam_as_bool1d(exits_i, f"{name} exits")
        if entries_i.size != n or exits_i.size != n:
            raise ValueError(
                f"{name} signal length mismatch: entries={entries_i.size}, "
                f"exits={exits_i.size}, expected={n}"
            )
        entries_or |= entries_i
        exits_or |= exits_i
    return entries_or, exits_or


def _raam_series_from_portfolio(pf, attr_name, close_index, label):
    """Align a 1-D portfolio series to the Close index."""
    fn = getattr(pf, attr_name, None)
    if not callable(fn):
        raise AttributeError(f"Portfolio has no callable {attr_name}()")
    values = fn()
    arr = np.asarray(values, dtype=float).reshape(-1)
    if arr.size != len(close_index):
        raise ValueError(
            f"{label} length mismatch: got {arr.size}, expected {len(close_index)}"
        )
    return pd.Series(arr, index=close_index, dtype=float)


def _raam_write_ticker_csv(frame, filepath):
    """Write one ticker file atomically (tmp then os.replace)."""
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"ticker frame must be a pandas DataFrame; got {type(frame).__name__}")
    if frame.empty:
        raise ValueError(f"ticker frame is empty; cannot save {filepath!r}")
    required = {"Date"}
    missing = required.difference(frame.columns)
    if missing:
        raise KeyError(f"ticker frame missing columns: {sorted(missing)}")
    if frame["Date"].isna().any():
        raise ValueError(f"ticker frame has missing Date values; cannot save {filepath!r}")
    if pd.Index(frame["Date"]).has_duplicates:
        raise ValueError(f"ticker frame has duplicate Date values; cannot save {filepath!r}")
    bnh_cols = [c for c in frame.columns if "BnH" in str(c)]
    strat_cols = [c for c in frame.columns if "Strategy" in str(c)]
    pos_cols = [c for c in frame.columns if "Position" in str(c)]
    if len(bnh_cols) != 1 or len(strat_cols) != 1 or len(pos_cols) != 1:
        raise KeyError(
            "ticker frame must have exactly one BnH, Strategy, and Position column; "
            f"got columns={list(frame.columns)}"
        )
    path = str(filepath).strip()
    if not path.lower().endswith(".csv"):
        raise ValueError(f"Unsupported file format (use .csv): {filepath!r}")
    parent = os.path.dirname(path)
    if parent and not os.path.isdir(parent):
        raise FileNotFoundError(
            f"Parent directory does not exist for filepath={path!r}: {parent}"
        )
    export_df = frame.replace([np.inf, -np.inf], np.nan)
    tmp_path = f"{path}.tmp"
    try:
        export_df.to_csv(tmp_path, index=False)
        os.replace(tmp_path, path)
    except PermissionError as exc:
        raise PermissionError(
            f"Permission denied writing {path!r}. Close the file if it is open elsewhere "
            f"and retry. Original error: {exc}"
        ) from exc
    except OSError as exc:
        raise OSError(f"Failed to write ticker file to {path!r}: {exc}") from exc
    finally:
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except OSError:
                pass
    return path


def _raam_session_kind(ticker):
    """Classify each ticker's trading session label: crypto / europe / na.

    Label only — daily allocation uses the prior native close for every session.
    """
    if ticker is None or (isinstance(ticker, float) and pd.isna(ticker)):
        raise ValueError(f"ticker for session kind must be a non-empty label; got {ticker!r}")
    t = str(ticker).strip()
    if not t:
        raise ValueError(f"ticker for session kind must be a non-empty label; got {ticker!r}")
    tu = t.upper()
    if "_is_crypto_by_ticker" in globals() and isinstance(_is_crypto_by_ticker, dict):
        crypto_map = {str(k).strip().upper(): bool(v) for k, v in _is_crypto_by_ticker.items()}
        if tu in crypto_map:
            return "crypto" if crypto_map[tu] else (
                "europe" if tu.endswith((".L", ".PA", ".AS", ".BR", ".DE", ".MI", ".SW", ".MC")) else "na"
            )
    if tu.endswith("-USD"):
        return "crypto"
    # Common European listing suffixes (LSE, Euronext, Xetra, etc.).
    if tu.endswith((".L", ".PA", ".AS", ".BR", ".DE", ".MI", ".SW", ".MC")):
        return "europe"
    return "na"


def _raam_tz_naive_index(index, label="index"):
    """Return a tz-naive DatetimeIndex; reject integers (epoch footgun) and NaT."""
    if index is None:
        raise TypeError(f"{label} is required")
    raw = pd.Index(index)
    if len(raw) < 1:
        raise ValueError(f"{label} is empty")
    if pd.api.types.is_integer_dtype(raw.dtype) or pd.api.types.is_float_dtype(raw.dtype):
        raise TypeError(
            f"{label} looks numeric ({raw.dtype}); refuse to interpret as unix timestamps. "
            f"Pass calendar dates."
        )
    if isinstance(index, pd.DatetimeIndex):
        idx = index.copy()
    else:
        idx = pd.DatetimeIndex(pd.to_datetime(raw, errors="coerce"))
    if idx.tz is not None:
        idx = idx.tz_convert("UTC").tz_localize(None)
    if bool(idx.isna().any()):
        raise ValueError(f"{label} contains unparseable dates")
    return idx


def _raam_require_native_position_series(native_pos, ticker):
    """Normalize and validate a close-tagged 0/1 native position series."""
    pos = pd.Series(native_pos).copy()
    if pos.empty:
        raise ValueError(f"Native position series is empty for {ticker!r}.")
    pos.index = _raam_tz_naive_index(pos.index, label=f"{ticker} native dates")
    if bool(pos.index.has_duplicates):
        raise ValueError(f"Native position index has duplicates for {ticker!r}.")
    pos = pos.sort_index()
    values = pd.to_numeric(pos, errors="coerce")
    arr = values.to_numpy(dtype=float)
    if bool((~np.isfinite(arr)).any()):
        raise ValueError(
            f"Native positions for {ticker!r} contain missing or non-finite values."
        )
    pos_ok = np.isclose(arr, 0.0) | np.isclose(arr, 1.0)
    if not bool(pos_ok.all()):
        raise ValueError(f"Native positions for {ticker!r} must be 0 or 1.")
    cleaned = pd.Series(np.where(np.isclose(arr, 1.0), 1.0, 0.0), index=pos.index, dtype=float)
    if len(cleaned) < 2:
        raise ValueError(
            f"Native position series for {ticker!r} needs at least 2 bars "
            f"so session-start lag is defined; got {len(cleaned)}."
        )
    return cleaned


def _raam_session_start_cutoff(index):
    """Last equity-calendar date whose native closes are treated as final."""
    idx = _raam_tz_naive_index(index, label="session-start cutoff index")
    return pd.Timestamp(idx[-1]).normalize()


def _raam_position_at_session_start(native_pos, cutoff, target_index, ticker):
    """
    Map close-tagged native positions onto a target calendar at session start.

    Position at native close T is applied at the start of the next native session
    (T+1), then asof-mapped onto the equity-session dates in target_index.
    Session kind is not used here: on daily bars every market uses the prior close.
    """
    pos = _raam_require_native_position_series(native_pos, ticker)

    cutoff_ts = pd.Timestamp(cutoff)
    if pd.isna(cutoff_ts):
        raise ValueError(f"Invalid session-start cutoff: {cutoff!r}")
    if cutoff_ts.tz is not None:
        cutoff_ts = cutoff_ts.tz_convert("UTC").tz_localize(None)
    cutoff_ts = cutoff_ts.normalize()
    pos = pos.loc[pos.index.normalize() <= cutoff_ts]
    if len(pos) < 2:
        raise ValueError(
            f"Need >= 2 native bars on/before cutoff {cutoff_ts.date()} for {ticker!r}; "
            f"got {len(pos)}."
        )

    # Close on T becomes the position in force at the start of session T+1.
    session_start = pos.shift(1)

    target = _raam_tz_naive_index(target_index, label=f"{ticker} target dates")
    if bool(pd.Index(target).has_duplicates):
        raise ValueError(f"target index has duplicates while mapping {ticker!r}.")
    if not target.is_monotonic_increasing:
        raise ValueError(f"target index must be sorted while mapping {ticker!r}.")

    aligned = (
        session_start.reindex(session_start.index.union(target))
        .sort_index()
        .ffill()
    )
    out = aligned.reindex(target)
    out.index = target_index
    return out.astype(float)


# ---------------------------------------------------------------------------
# Load master leaderboard and write one file per ticker
# ---------------------------------------------------------------------------
_leaderboard_path = _raam_leaderboard_filepath()
try:
    leaderboard_df = pd.read_json(_leaderboard_path, orient="records")
except ImportError as exc:
    raise ImportError(
        f"Reading 'json' requires a dependency that is not installed "
        f"(pip install pandas). Original error: {exc}"
    ) from exc
except ValueError as exc:
    raise ValueError(
        f"Invalid leaderboard JSON (expected a records array) at {_leaderboard_path!r}: {exc}"
    ) from exc
except OSError as exc:
    raise OSError(f"Failed to read leaderboard JSON {_leaderboard_path!r}: {exc}") from exc

if not isinstance(leaderboard_df, pd.DataFrame):
    raise TypeError(
        f"Leaderboard JSON must deserialize to a DataFrame; got {type(leaderboard_df).__name__}"
    )
if leaderboard_df.empty:
    raise ValueError(f"Leaderboard JSON is empty: {_leaderboard_path}")
required_cols = {"ticker", "ensemble", "members", "param_display"}
missing_cols = required_cols.difference(leaderboard_df.columns)
if missing_cols:
    raise KeyError(f"Leaderboard missing columns required to build ticker files: {sorted(missing_cols)}")

_leaderboard_tickers = []
for loc_idx, row in leaderboard_df.iterrows():
    try:
        _leaderboard_tickers.append(_raam_require_label(row["ticker"], "ticker"))
    except ValueError as exc:
        raise ValueError(f"Invalid ticker on leaderboard row {loc_idx}: {exc}") from exc
_ticker_series = pd.Series(_leaderboard_tickers)
if bool(_ticker_series.duplicated().any()):
    dupes = _ticker_series[_ticker_series.duplicated()].tolist()
    raise ValueError(
        "leaderboard has duplicate ticker(s): "
        + ", ".join(dict.fromkeys(str(t) for t in dupes))
    )

_raam_dir = _raam_data_dir()

print("=" * 80)
print(f"DATA PREPARATION: WRITING PER-TICKER FILES FROM {os.path.basename(_leaderboard_path)}")
print(f"Tickers in leaderboard: {len(leaderboard_df)}")
print("=" * 80)

raam_tickers = []
raam_filepaths = []
raam_native_positions = {}
raam_failures = []
_saved_ticker = globals().get("TICKER")
_saved_ticker_was_set = "TICKER" in globals()

try:
    for loc_idx, row in leaderboard_df.iterrows():
        ticker = None
        try:
            ticker = _raam_require_label(row["ticker"], "ticker")
            members = _raam_members_cell(row["members"])
            param_display = _raam_require_label(row["param_display"], "param_display")
            if "n_members" in leaderboard_df.columns:
                n_members = safe_float(row["n_members"])
                if not np.isfinite(n_members) or abs(n_members - len(members)) > 1e-9:
                    raise ValueError(
                        f"n_members={row['n_members']!r} does not match members={members!r}"
                    )
            if "combine_rule" in leaderboard_df.columns:
                if not pd.notna(row.get("combine_rule")):
                    raise ValueError("combine_rule is missing")
                combine_rule = _raam_require_label(row["combine_rule"], "combine_rule").upper()
                if combine_rule != "OR":
                    raise ValueError(
                        f"unsupported combine_rule={row['combine_rule']!r} (expected OR)"
                    )

            if ticker not in close_by_ticker:
                raise KeyError(
                    f"No prepared Close series for ticker {ticker!r}. "
                    f"Known: {sorted(close_by_ticker)}"
                )

            # Global TICKER is required by OHLC helpers (ADX / SuperTrend / Vortex / Donchian).
            TICKER = ticker
            close = close_by_ticker[ticker]
            if close is None or len(close) < 2:
                raise ValueError(f"Close series for TICKER={ticker!r} is missing or too short.")

            member_params = _raam_member_params_from_param_display(members, param_display)
            signal_map = {}
            for indicator_name in members:
                spec = get_active_indicator_spec(indicator_name)
                build_signals = spec.get("build_signals") if isinstance(spec, dict) else None
                if not callable(build_signals):
                    raise KeyError(f"{indicator_name} spec is missing callable build_signals")
                entries, exits = build_signals(
                    close, member_params[indicator_name], shift_signals=True
                )
                if len(entries) != len(close) or len(exits) != len(close):
                    raise ValueError(
                        f"{indicator_name} signal length mismatch "
                        f"(entries={len(entries)}, exits={len(exits)}, close={len(close)})"
                    )
                signal_map[indicator_name] = (
                    pd.Series(np.asarray(entries, dtype=bool), index=close.index, dtype=bool),
                    pd.Series(np.asarray(exits, dtype=bool), index=close.index, dtype=bool),
                )

            entries_or, exits_or = _raam_or_combine_member_signals(signal_map, members)
            pf = run_signal_backtest(close, entries_or, exits_or)
            strategy_returns = _raam_series_from_portfolio(
                pf, "returns", close.index, f"{ticker} Strategy"
            )
            position = _raam_series_from_portfolio(
                pf, "position_mask", close.index, f"{ticker} Position"
            )
            position = (position != 0).astype(float)
            bnh_returns = pd.Series(close.pct_change(), index=close.index, dtype=float)

            dates = _raam_tz_naive_index(close.index, label=f"{ticker} close dates")
            native_pos = _raam_require_native_position_series(
                pd.Series(position.to_numpy(dtype=float), index=dates, dtype=float),
                ticker,
            )
            raam_native_positions[ticker] = native_pos
            ticker_df = pd.DataFrame({
                "Date": dates,
                f"{ticker}_BnH": bnh_returns.to_numpy(dtype=float),
                f"{ticker}_Strategy": strategy_returns.to_numpy(dtype=float),
                f"{ticker}_Position": native_pos.to_numpy(dtype=float),
            })
            filepath = _raam_write_ticker_csv(ticker_df, _raam_ticker_filepath(ticker, _raam_dir))
            raam_tickers.append(ticker)
            raam_filepaths.append(filepath)
            print(
                f"  SAVED {ticker}: {os.path.basename(filepath)}  "
                f"rows={len(ticker_df)}  members={members}  "
                f"session={_raam_session_kind(ticker)}"
            )
        except _RAAM_TICKER_ERRORS as exc:
            label = ticker if ticker else f"row {loc_idx}"
            raam_failures.append(f"{label}: {type(exc).__name__}: {exc}")
            print(f"  SKIP {label}: {type(exc).__name__}: {exc}")
            continue

    if raam_failures:
        print(f"Ticker file failures: {len(raam_failures)}")
        for _detail in raam_failures:
            print(f"  - {_detail}")

    if not raam_tickers:
        raise ValueError(
            "No per-ticker RAAM files were written. "
            f"Leaderboard={_leaderboard_path!r}."
        )
    missing_native = [t for t in raam_tickers if t not in raam_native_positions]
    if missing_native:
        raise KeyError(f"Native position series missing after write for: {missing_native}")

    print(f"Saved {len(raam_tickers)} ticker file(s) to {_raam_dir}")
    print(
        "Native session map: "
        + ", ".join(f"{t}={_raam_session_kind(t)}" for t in raam_tickers)
    )
finally:
    if _saved_ticker_was_set:
        TICKER = _saved_ticker

DATA PREPARATION: WRITING PER-TICKER FILES FROM master_leaderboard_ensembles.json
Tickers in leaderboard: 6
  SAVED QTUM: qtum_raam.csv  rows=1447  members=['Triple EMA', 'VORTEX', 'ALMA']  session=na
  SAVED BTC-USD: btc_usd_raam.csv  rows=2106  members=['Triple EMA', 'STC', 'ADX']  session=crypto
  SAVED TQQQ: tqqq_raam.csv  rows=1447  members=['MACD', 'STC', 'RSI']  session=na
  SAVED SMH.L: smh_l_raam.csv  rows=1453  members=['Triple EMA', 'MACD', 'RSI']  session=europe
  SAVED UPRO: upro_raam.csv  rows=1447  members=['STC', 'RSI', 'ALMA']  session=na
  SAVED GLD: gld_raam.csv  rows=1447  members=['STC', 'TRIX', 'ALMA']  session=na
Saved 6 ticker file(s) to c:\Quant Rick's Trading Academy\RAAM_data
Native session map: QTUM=na, BTC-USD=crypto, TQQQ=na, SMH.L=europe, UPRO=na, GLD=na


In [7]:
# LOAD PER-TICKER FILES
# Each file has Date, a BnH column, a Strategy column, and a Position column.
# LIVE TRADING: outer-align on equity session dates so one market holiday
# (e.g. SMH.L missing 2026-09-14) does not freeze signals on an older day.

import os

if "raam_tickers" not in globals() or "raam_filepaths" not in globals():
    raise NameError(
        "raam_tickers / raam_filepaths are missing. Run the per-ticker file cell first."
    )
if not raam_tickers or not raam_filepaths:
    raise ValueError("raam_tickers / raam_filepaths are empty. Run the per-ticker file cell first.")
if len(raam_tickers) != len(raam_filepaths):
    raise ValueError(
        f"raam_tickers length ({len(raam_tickers)}) != raam_filepaths length ({len(raam_filepaths)})."
    )
_session_helpers = (
    "_raam_session_kind",
    "_raam_session_start_cutoff",
    "_raam_position_at_session_start",
    "_raam_require_native_position_series",
)
_missing_session_helpers = [
    name for name in _session_helpers
    if name not in globals() or not callable(globals()[name])
]
if _missing_session_helpers:
    raise NameError(
        "Session helpers missing: "
        + ", ".join(_missing_session_helpers)
        + ". Re-run the per-ticker file cell first."
    )

_read_csv_errors = (ValueError, OSError)
if hasattr(pd.errors, "ParserError"):
    _read_csv_errors = _read_csv_errors + (pd.errors.ParserError,)


def _raam_require_parsed_dates(frame, ticker, filepath):
    """Require a unique, parseable Date column (used as the merge key)."""
    if "Date" not in frame.columns:
        raise KeyError(f"{filepath!r} is missing required column 'Date'")
    dates = pd.to_datetime(frame["Date"], errors="coerce")
    if dates.isna().any():
        raise ValueError(f"{filepath!r} has unparseable Date values for ticker {ticker!r}")
    if dates.duplicated().any():
        raise ValueError(f"{filepath!r} has duplicate Date values for ticker {ticker!r}")
    frame = frame.copy()
    frame["Date"] = dates
    return frame


dfs = []
tickers = []
_existing_native = (
    raam_native_positions
    if isinstance(globals().get("raam_native_positions"), dict)
    else {}
)
raam_native_positions = {}
for ticker, filepath in zip(raam_tickers, raam_filepaths):
    ticker = str(ticker)
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"Per-ticker file is missing for {ticker!r}: {filepath!r}")
    try:
        df = pd.read_csv(filepath, parse_dates=["Date"])
    except _read_csv_errors as exc:
        raise ValueError(f"Failed to read {filepath!r} for ticker {ticker!r}: {exc}") from exc
    if df is None or df.empty:
        raise ValueError(f"Per-ticker file is empty for {ticker!r}: {filepath!r}")
    df = _raam_require_parsed_dates(df, ticker, filepath)
    pos_cols = [col for col in df.columns if "Position" in str(col)]
    if len(pos_cols) != 1:
        raise KeyError(
            f"{filepath!r} must have exactly one Position column; got {list(df.columns)}"
        )
    # Reuse write-cell series when present; otherwise rebuild from the CSV column.
    prior = _existing_native.get(ticker, _existing_native.get(str(ticker)))
    if isinstance(prior, pd.Series) and len(prior) == len(df):
        native_pos = _raam_require_native_position_series(prior, ticker)
    else:
        native_pos = _raam_require_native_position_series(
            pd.Series(
                df[pos_cols[0]].to_numpy(),
                index=pd.DatetimeIndex(df["Date"]),
            ),
            ticker,
        )
    raam_native_positions[ticker] = native_pos
    dfs.append(df)
    tickers.append(ticker)

if not dfs:
    raise ValueError("No per-ticker files were loaded.")
if len(raam_native_positions) != len(tickers):
    raise ValueError("Failed to rebuild raam_native_positions for every loaded ticker.")

# Detect 24/7 names so weekend-only crypto bars do not create equity signal dates.
if "_is_crypto_by_ticker" in globals() and isinstance(_is_crypto_by_ticker, dict):
    crypto_flags = {
        t: bool(_is_crypto_by_ticker.get(t, str(t).upper().endswith("-USD")))
        for t in tickers
    }
else:
    crypto_flags = {t: str(t).upper().endswith("-USD") for t in tickers}
equity_tickers = [t for t in tickers if not crypto_flags.get(t, False)]
if not equity_tickers:
    equity_tickers = list(tickers)

# BUILD ALIGNED BUY & HOLD RETURNS (outer join on equity session dates)
bnh_cols0 = [col for col in dfs[0].columns if "BnH" in str(col)]
if len(bnh_cols0) != 1:
    raise KeyError(f"{tickers[0]!r} must have exactly one BnH column; got {list(dfs[0].columns)}")
bnh_df = dfs[0][["Date", bnh_cols0[0]]].copy().rename(columns={bnh_cols0[0]: tickers[0]})
for df, ticker in zip(dfs[1:], tickers[1:]):
    bnh_cols = [col for col in df.columns if "BnH" in str(col)]
    if len(bnh_cols) != 1:
        raise KeyError(
            f"{ticker!r} must have exactly one BnH column; got {list(df.columns)}"
        )
    bnh_col = bnh_cols[0]
    bnh_df = bnh_df.merge(df[["Date", bnh_col]], on="Date", how="outer")
    bnh_df = bnh_df.rename(columns={bnh_col: ticker})

bnh_df = bnh_df.sort_values("Date")
if bnh_df["Date"].duplicated().any():
    raise ValueError("Buy & Hold outer merge produced duplicate dates.")
bnh_df = bnh_df.set_index("Date").replace([np.inf, -np.inf], np.nan)

# Keep dates where a majority of equity tickers actually printed a bar.
equity_present = bnh_df[equity_tickers].notna()
min_equity = max(1, int(np.ceil(len(equity_tickers) / 2.0)))
keep_dates = equity_present.sum(axis=1) >= min_equity
bnh_df = bnh_df.loc[keep_dates].copy()
if len(bnh_df) < 2:
    raise ValueError(
        f"Buy & Hold alignment produced too few equity-session dates ({len(bnh_df)}). "
        f"Need >= {min_equity} of {equity_tickers} on a date."
    )

# After every ticker has started, missing BnH on a holiday = 0% (flat).
started = bnh_df.notna().cummax().all(axis=1)
if not bool(started.any()):
    raise ValueError("Buy & Hold alignment never reaches a date where all tickers have started.")
bnh_df = bnh_df.loc[started].copy()
filled_bnh = bnh_df.isna()
bnh_df = bnh_df.fillna(0.0)

print(
    f"Aligned Buy & Hold returns: {len(bnh_df)} equity-session dates | "
    f"tickers={list(bnh_df.columns)}"
)
print(f"Date range: {bnh_df.index.min().date()} -> {bnh_df.index.max().date()}")
print(
    f"Alignment rule: outer join, keep dates with >= {min_equity}/{len(equity_tickers)} "
    f"equity prints; holiday gaps filled with 0% BnH."
)
if pd.Timestamp("2026-09-14") in bnh_df.index:
    print("OK: 2026-09-14 is present in the aligned calendar.")
else:
    print("WARNING: 2026-09-14 is still missing from the aligned calendar.")
recent_fill = filled_bnh.tail(8)
if bool(recent_fill.any().any()):
    print("Recent holiday/gap fills (True = filled 0% BnH):")
    print(recent_fill)
bnh_df.tail()

Aligned Buy & Hold returns: 1446 equity-session dates | tickers=['QTUM', 'BTC-USD', 'TQQQ', 'SMH.L', 'UPRO', 'GLD']
Date range: 2021-01-05 -> 2026-10-07
Alignment rule: outer join, keep dates with >= 3/5 equity prints; holiday gaps filled with 0% BnH.
OK: 2026-09-14 is present in the aligned calendar.
Recent holiday/gap fills (True = filled 0% BnH):
             QTUM  BTC-USD   TQQQ  SMH.L   UPRO    GLD
Date                                                  
2026-09-28  False    False  False  False  False  False
2026-09-29  False    False  False  False  False  False
2026-09-30  False    False  False  False  False  False
2026-10-01  False    False  False  False  False  False
2026-10-02  False    False  False  False  False  False
2026-10-05  False    False  False  False  False  False
2026-10-06  False    False  False  False  False  False
2026-10-07  False     True  False   True  False  False


,QTUM,BTC-USD,TQQQ,SMH.L,UPRO,GLD
Date,,,,,,
2026-10-01,0.010751,0.015550,0.008971,-0.001922,0.005307,0.005042
2026-10-02,0.018336,-0.004194,0.028960,0.036064,0.021184,-0.006845
2026-10-05,0.002179,-0.008022,0.026046,-0.002028,0.019950,-0.001552
2026-10-06,0.006330,-0.002670,0.013595,0.005588,0.016375,0.007166
2026-10-07,-0.018743,0.000000,-0.007478,0.000000,-0.007992,-0.016716


In [8]:
# BUILD ALIGNED STRATEGY RETURNS AND POSITIONS
# LIVE TRADING: same equity-session outer alignment as Buy & Hold.
# Missing Position on a holiday is forward-filled (keep last known ensemble state).

if "dfs" not in globals() or "tickers" not in globals():
    raise NameError("dfs / tickers are missing. Run the per-ticker load cell first.")
if not dfs or not tickers:
    raise ValueError("dfs / tickers are empty. Run the per-ticker load cell first.")
if len(dfs) != len(tickers):
    raise ValueError(f"dfs length ({len(dfs)}) != tickers length ({len(tickers)}).")
if "bnh_df" not in globals() or not isinstance(bnh_df, pd.DataFrame) or bnh_df.empty:
    raise ValueError("bnh_df is missing or empty. Run the Buy & Hold alignment cell first.")


def _raam_require_strat_pos_cols(frame, ticker):
    strat_cols = [col for col in frame.columns if "Strategy" in str(col)]
    pos_cols = [col for col in frame.columns if "Position" in str(col)]
    if len(strat_cols) != 1 or len(pos_cols) != 1:
        raise KeyError(
            f"{ticker!r} must have exactly one Strategy column and one Position column; "
            f"got {list(frame.columns)}"
        )
    return strat_cols[0], pos_cols[0]


if "Date" not in dfs[0].columns:
    raise KeyError(f"{tickers[0]!r} is missing required column 'Date'")
strat_col0, pos_col0 = _raam_require_strat_pos_cols(dfs[0], tickers[0])
strat_df = (
    dfs[0][["Date", strat_col0, pos_col0]]
    .copy()
    .rename(
        columns={
            strat_col0: f"{tickers[0]}_Return",
            pos_col0: f"{tickers[0]}_Position",
        }
    )
)
for df, ticker in zip(dfs[1:], tickers[1:]):
    if "Date" not in df.columns:
        raise KeyError(f"{ticker!r} is missing required column 'Date'")
    strat_col, pos_col = _raam_require_strat_pos_cols(df, ticker)
    strat_df = strat_df.merge(df[["Date", strat_col, pos_col]], on="Date", how="outer")
    strat_df = strat_df.rename(
        columns={
            strat_col: f"{ticker}_Return",
            pos_col: f"{ticker}_Position",
        }
    )

strat_df = strat_df.sort_values("Date")
if strat_df["Date"].duplicated().any():
    raise ValueError("Strategy/Position outer merge produced duplicate dates.")
strat_df = strat_df.set_index("Date").replace([np.inf, -np.inf], np.nan)

# Use the same equity-session index as bnh_df so ranks/positions stay aligned.
strat_df = strat_df.reindex(bnh_df.index)
ret_cols = [f"{ticker}_Return" for ticker in tickers]
pos_cols = [f"{ticker}_Position" for ticker in tickers]
missing = [c for c in ret_cols + pos_cols if c not in strat_df.columns]
if missing:
    raise KeyError(f"strat_df missing columns after alignment: {missing}")

filled_pos = strat_df[pos_cols].isna()
# Holiday gap: carry last ensemble position; missing strategy return = 0.
strat_df[pos_cols] = strat_df[pos_cols].ffill()
strat_df[pos_cols] = strat_df[pos_cols].fillna(0.0)
strat_df[ret_cols] = strat_df[ret_cols].fillna(0.0)

if len(strat_df) < 2:
    raise ValueError(
        f"Strategy/Position alignment produced too few dates ({len(strat_df)})."
    )
if not strat_df.index.equals(bnh_df.index):
    raise ValueError("strat_df and bnh_df indexes do not match after alignment.")

print(
    f"Aligned Strategy returns & Positions: {len(strat_df)} dates | "
    f"columns={len(strat_df.columns)}"
)
print(f"Date range: {strat_df.index.min().date()} -> {strat_df.index.max().date()}")
recent_pos_fill = filled_pos.tail(8)
if bool(recent_pos_fill.any().any()):
    print("Recent holiday/gap position fills (True = forward-filled):")
    print(recent_pos_fill)
strat_df.tail()

Aligned Strategy returns & Positions: 1446 dates | columns=12
Date range: 2021-01-05 -> 2026-10-07
Recent holiday/gap position fills (True = forward-filled):
            QTUM_Position  BTC-USD_Position  TQQQ_Position  SMH.L_Position  \
Date                                                                         
2026-09-28          False             False          False           False   
2026-09-29          False             False          False           False   
2026-09-30          False             False          False           False   
2026-10-01          False             False          False           False   
2026-10-02          False             False          False           False   
2026-10-05          False             False          False           False   
2026-10-06          False             False          False           False   
2026-10-07          False              True          False            True   

            UPRO_Position  GLD_Position  
Date               

,QTUM_Return,QTUM_Position,BTC-USD_Return,BTC-USD_Position,TQQQ_Return,TQQQ_Position,SMH.L_Return,SMH.L_Position,UPRO_Return,UPRO_Position,GLD_Return,GLD_Position
Date,,,,,,,,,,,,
2026-10-01,0.010751,1.0,0.015550,1.0,0.008971,1.0,-0.001922,1.0,0.0,0.0,0.0,0.0
2026-10-02,0.018336,1.0,-0.004194,1.0,0.028960,1.0,0.036064,1.0,0.0,0.0,0.0,0.0
2026-10-05,0.002179,1.0,-0.008022,1.0,0.026046,1.0,-0.002028,1.0,0.0,0.0,0.0,0.0
2026-10-06,0.006330,1.0,-0.002670,1.0,0.013595,1.0,0.005588,1.0,0.0,0.0,0.0,0.0
2026-10-07,-0.018743,1.0,0.000000,1.0,-0.007478,1.0,0.000000,1.0,0.0,0.0,0.0,0.0


SIGNAL GENERATION - RSI ON CUMULATIVE RETURNS AND RANKING
---------------------------------------------------------

This section converts aligned Buy & Hold returns into ranked allocation signals.

- RSI is applied to each ticker's cumulative Buy & Hold return series (not price), so it measures the strength of the performance trend.
- Assets with RSI below 50 are excluded (rank 0).
- Remaining assets are ranked by RSI; rank 1 is the strongest.

---

In [9]:
# APPLY RSI TO BUY & HOLD CUMULATIVE RETURNS
# RSI measures the strength of the performance trend, not price.

if "talib" not in globals():
    raise NameError("talib is missing. Run the import cell first.")
if "bnh_df" not in globals():
    raise NameError("bnh_df is missing. Run the Buy & Hold alignment cell first.")
if not isinstance(bnh_df, pd.DataFrame):
    raise TypeError(f"bnh_df must be a pandas DataFrame; got {type(bnh_df).__name__}")
if bnh_df.empty:
    raise ValueError("bnh_df is empty. Run the Buy & Hold alignment cell first.")
if bnh_df.shape[1] < 1:
    raise ValueError("bnh_df has no ticker columns to rank.")
if bnh_df.columns.duplicated().any():
    dupes = bnh_df.columns[bnh_df.columns.duplicated()].tolist()
    raise ValueError(f"bnh_df has duplicate ticker column(s): {dupes}")
if bnh_df.index.has_duplicates:
    raise ValueError("bnh_df has duplicate dates; cannot build a cumulative return series.")

non_numeric = [
    col for col in bnh_df.columns if not np.issubdtype(bnh_df[col].dtype, np.number)
]
if non_numeric:
    raise TypeError(f"bnh_df columns must be numeric; got non-numeric: {non_numeric}")

RSI_TIMEPERIOD = 28
RSI_EXCLUDE_LEVEL = 50

try:
    rsi_timeperiod_num = float(RSI_TIMEPERIOD)
    rsi_exclude_num = float(RSI_EXCLUDE_LEVEL)
except (TypeError, ValueError) as exc:
    raise ValueError(
        f"RSI_TIMEPERIOD and RSI_EXCLUDE_LEVEL must be numeric; "
        f"got {RSI_TIMEPERIOD!r}, {RSI_EXCLUDE_LEVEL!r}"
    ) from exc
if not np.isfinite(rsi_timeperiod_num) or rsi_timeperiod_num != int(rsi_timeperiod_num):
    raise ValueError(f"RSI_TIMEPERIOD must be a whole number; got {RSI_TIMEPERIOD!r}")
RSI_TIMEPERIOD = int(rsi_timeperiod_num)
if RSI_TIMEPERIOD <= 0:
    raise ValueError(f"RSI_TIMEPERIOD must be a positive integer; got {RSI_TIMEPERIOD!r}")
if not np.isfinite(rsi_exclude_num) or rsi_exclude_num < 0 or rsi_exclude_num > 100:
    raise ValueError(
        f"RSI_EXCLUDE_LEVEL must be a finite number in [0, 100]; got {RSI_EXCLUDE_LEVEL!r}"
    )
RSI_EXCLUDE_LEVEL = float(rsi_exclude_num)

if len(bnh_df) < RSI_TIMEPERIOD + 1:
    raise ValueError(
        f"Not enough overlapping dates ({len(bnh_df)}) to compute RSI({RSI_TIMEPERIOD})."
    )

# Daily returns -> wealth index. Leading NaNs (typical first-day pct_change) start
# as a 0 return. Interior missing values are not treated as 0%.
bnh_returns = bnh_df.replace([np.inf, -np.inf], np.nan).astype(float)
if not bool(bnh_returns.notna().any().all()):
    empty_cols = [col for col in bnh_returns.columns if not bnh_returns[col].notna().any()]
    raise ValueError(f"No finite Buy & Hold returns for: {empty_cols}")
if bool((bnh_returns <= -1.0).any().any()):
    bad_cols = [col for col in bnh_returns.columns if bool((bnh_returns[col] <= -1.0).any())]
    raise ValueError(
        f"Buy & Hold returns <= -1 would collapse the wealth index; columns: {bad_cols}"
    )

started = bnh_returns.notna().cummax()
interior_nan = bnh_returns.isna() & started
if bool(interior_nan.any().any()):
    bad_cols = [col for col in bnh_returns.columns if bool(interior_nan[col].any())]
    raise ValueError(
        f"Interior missing Buy & Hold returns; cannot build a cumulative series: {bad_cols}"
    )
bnh_returns = bnh_returns.mask(~started, 0.0)

bnh_cum = (1.0 + bnh_returns).cumprod()
if not np.isfinite(bnh_cum.to_numpy(dtype=float)).any():
    raise ValueError("Cumulative Buy & Hold series contains no finite values.")

bnh_df_rsi = bnh_cum.copy()

print(f"Calculating RSI ({RSI_TIMEPERIOD}-period) for each ticker...")
for ticker in bnh_cum.columns:
    cum_returns = bnh_cum[ticker].to_numpy(dtype=float)
    if np.isfinite(cum_returns).sum() < RSI_TIMEPERIOD + 1:
        raise ValueError(
            f"Not enough finite cumulative returns to compute RSI for {ticker!r}."
        )
    try:
        rsi = talib.RSI(cum_returns, timeperiod=RSI_TIMEPERIOD)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"RSI failed for {ticker!r}: {exc}") from exc
    rsi = np.asarray(rsi, dtype=float).reshape(-1)
    if rsi.size != len(bnh_cum):
        raise ValueError(
            f"RSI length mismatch for {ticker!r}: got {rsi.size}, expected {len(bnh_cum)}"
        )
    if np.isfinite(rsi).sum() == 0:
        raise ValueError(f"RSI could not be computed for {ticker!r}.")
    bnh_df_rsi[f"{ticker}_RSI"] = rsi

print(f"RSI calculated for {len(bnh_cum.columns)} tickers")
print(bnh_df_rsi.tail())

Calculating RSI (28-period) for each ticker...
RSI calculated for 6 tickers
                QTUM   BTC-USD      TQQQ     SMH.L      UPRO       GLD  \
Date                                                                     
2026-10-01  3.868666  1.715565  3.809178  4.907397  4.173372  2.099271   
2026-10-02  3.939602  1.708369  3.919491  5.084380  4.261782  2.084901   
2026-10-05  3.948185  1.694665  4.021579  5.074070  4.346803  2.081665   
2026-10-06  3.973177  1.690141  4.076251  5.102422  4.417983  2.096583   
2026-10-07  3.898707  1.690141  4.045770  5.102422  4.382676  2.061537   

             QTUM_RSI  BTC-USD_RSI   TQQQ_RSI  SMH.L_RSI   UPRO_RSI    GLD_RSI  
Date                                                                            
2026-10-01  53.917996    62.344520  55.864183  54.703218  50.688223  45.013259  
2026-10-02  56.121040    61.706264  57.849526  57.864559  53.051690  44.123729  
2026-10-05  56.382686    60.483299  59.593876  57.621622  55.193377  43.921027  


In [10]:
# RANKING SYSTEM BASED ON RSI
# Rank 1 = highest RSI. Rank 0 = RSI below 50 (excluded).

if "bnh_df_rsi" not in globals():
    raise NameError("bnh_df_rsi is missing. Run the RSI signal-generation cell first.")
if not isinstance(bnh_df_rsi, pd.DataFrame):
    raise TypeError(f"bnh_df_rsi must be a pandas DataFrame; got {type(bnh_df_rsi).__name__}")
if bnh_df_rsi.empty:
    raise ValueError("bnh_df_rsi is empty. Run the RSI signal-generation cell first.")
if bnh_df_rsi.index.has_duplicates:
    raise ValueError("bnh_df_rsi has duplicate dates; cannot rank by date.")

rsi_cols = [col for col in bnh_df_rsi.columns if str(col).endswith("_RSI")]
if not rsi_cols:
    raise KeyError("bnh_df_rsi has no '_RSI' columns. Run the RSI signal-generation cell first.")

tickers_from_rsi = []
for col in rsi_cols:
    ticker = str(col)[: -len("_RSI")]
    if not ticker:
        raise ValueError(f"Invalid RSI column name: {col!r}")
    tickers_from_rsi.append(ticker)
if len(set(tickers_from_rsi)) != len(tickers_from_rsi):
    raise ValueError(f"Duplicate RSI ticker(s): {tickers_from_rsi}")

if "RSI_EXCLUDE_LEVEL" not in globals():
    raise NameError("RSI_EXCLUDE_LEVEL is missing. Run the RSI signal-generation cell first.")
try:
    rsi_exclude = float(RSI_EXCLUDE_LEVEL)
except (TypeError, ValueError) as exc:
    raise ValueError(f"RSI_EXCLUDE_LEVEL must be numeric; got {RSI_EXCLUDE_LEVEL!r}") from exc
if not np.isfinite(rsi_exclude) or rsi_exclude < 0 or rsi_exclude > 100:
    raise ValueError(
        f"RSI_EXCLUDE_LEVEL must be a finite number in [0, 100]; got {RSI_EXCLUDE_LEVEL!r}"
    )
RSI_EXCLUDE_LEVEL = rsi_exclude

try:
    rsi_df = bnh_df_rsi.loc[:, rsi_cols].astype(float)
except (TypeError, ValueError) as exc:
    raise TypeError(f"RSI columns must be numeric: {exc}") from exc
if bool(rsi_df.isna().all().all()):
    raise ValueError("All RSI values are missing; cannot rank assets.")

# Filter first (RSI below threshold / non-finite -> excluded), then rank
# remaining assets. Rank 1 = strongest among those still eligible that day.
eligible = rsi_df.ge(RSI_EXCLUDE_LEVEL)
ranking_df = rsi_df.where(eligible).rank(axis=1, method="first", ascending=False)
ranking_df = ranking_df.fillna(0).astype(int)
ranking_df.columns = [f"{ticker}_Rank" for ticker in tickers_from_rsi]

print("=" * 80)
print("RSI-Based Ranking System")
print("=" * 80)
print("Rank 1 = Highest RSI value")
print(f"Rank 0 = RSI below {RSI_EXCLUDE_LEVEL:g} (excluded)")
print(f"Total tickers processed: {len(rsi_cols)}")
print(f"Date range: {ranking_df.index.min()} to {ranking_df.index.max()}")
print(ranking_df.tail(60))

RSI-Based Ranking System
Rank 1 = Highest RSI value
Rank 0 = RSI below 50 (excluded)
Total tickers processed: 6
Date range: 2021-01-05 00:00:00 to 2026-10-07 00:00:00
            QTUM_Rank  BTC-USD_Rank  TQQQ_Rank  SMH.L_Rank  UPRO_Rank  \
Date                                                                    
2026-07-15          0             0          2           3          1   
2026-07-16          0             0          0           0          1   
2026-07-17          0             0          0           0          1   
2026-07-20          0             0          0           0          1   
2026-07-21          0             0          0           2          1   
2026-07-22          0             0          0           2          1   
2026-07-23          0             0          0           1          0   
2026-07-24          0             0          0           0          0   
2026-07-27          0             0          0           0          0   
2026-07-28          0         

STRATEGY EXECUTION - DYNAMIC ALLOCATION
---------------------------------------

This section turns lagged ranks and ensemble position signals into portfolio weights.

- `TOP_N` is the number of slots; each selected asset receives weight `1 / TOP_N`.
- Dual eligibility: yesterday's rank is in `1..TOP_N` and the position known at the start of that ticker's own session is 1.
- Leftover slots go to `SAFE_HAVEN` when it is eligible (yesterday's rank > 0 and session-start position == 1); otherwise leftover is cash.
- The live-trading cell then converts the latest ranks/positions into next-session target weights.

---

In [11]:
# STRATEGY EXECUTION: PARAMETERS AND COMBINED FRAME
# TOP_N slots at 1/TOP_N each. SAFE_HAVEN receives leftover eligible slots.

if "ranking_df" not in globals():
    raise NameError("ranking_df is missing. Run the ranking cell first.")
if "strat_df" not in globals():
    raise NameError("strat_df is missing. Run the Strategy/Position alignment cell first.")
if "bnh_df" not in globals():
    raise NameError("bnh_df is missing. Run the Buy & Hold correlation cell first.")
for name, frame in (("ranking_df", ranking_df), ("strat_df", strat_df), ("bnh_df", bnh_df)):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame; got {type(frame).__name__}")
    if frame.empty:
        raise ValueError(f"{name} is empty.")
    if frame.index.has_duplicates:
        raise ValueError(f"{name} has duplicate dates; cannot allocate by date.")
    if pd.isna(frame.index).any():
        raise ValueError(f"{name} has missing dates; cannot allocate by date.")
    if not frame.index.is_monotonic_increasing:
        raise ValueError(f"{name} is not sorted by date; a 1-day lag would be wrong.")

if not ranking_df.index.equals(strat_df.index) or not ranking_df.index.equals(bnh_df.index):
    raise ValueError(
        "ranking_df, strat_df, and bnh_df must share the same Date index. "
        "Re-run the correlation and ranking cells."
    )

TOP_N = 4
SAFE_HAVEN = "GLD"

try:
    top_n_num = float(TOP_N)
except (TypeError, ValueError) as exc:
    raise ValueError(f"TOP_N must be numeric; got {TOP_N!r}") from exc
if not np.isfinite(top_n_num) or top_n_num != int(top_n_num):
    raise ValueError(f"TOP_N must be a whole number; got {TOP_N!r}")
TOP_N = int(top_n_num)
if TOP_N < 1:
    raise ValueError(f"TOP_N must be a positive integer; got {TOP_N!r}")

if SAFE_HAVEN is None or not str(SAFE_HAVEN).strip():
    raise ValueError(f"SAFE_HAVEN must be a non-empty ticker; got {SAFE_HAVEN!r}")
SAFE_HAVEN = str(SAFE_HAVEN).strip()

rank_cols = [col for col in ranking_df.columns if str(col).endswith("_Rank")]
if not rank_cols:
    raise KeyError("ranking_df has no '_Rank' columns. Run the ranking cell first.")

tickers = []
for col in rank_cols:
    ticker = str(col)[: -len("_Rank")]
    if not ticker:
        raise ValueError(f"Invalid rank column name: {col!r}")
    tickers.append(ticker)
if len(set(tickers)) != len(tickers):
    raise ValueError(f"Duplicate rank ticker(s): {tickers}")
if SAFE_HAVEN not in tickers:
    raise KeyError(
        f"SAFE_HAVEN={SAFE_HAVEN!r} is not in the ranked universe {tickers}."
    )
if TOP_N > len(tickers):
    raise ValueError(
        f"TOP_N={TOP_N} is greater than the number of tickers ({len(tickers)})."
    )

missing_pos = [f"{ticker}_Position" for ticker in tickers if f"{ticker}_Position" not in strat_df.columns]
missing_ret = [ticker for ticker in tickers if ticker not in bnh_df.columns]
if missing_pos or missing_ret:
    raise KeyError(
        "Missing allocation inputs: "
        f"position={missing_pos}, bnh={missing_ret}"
    )

rank_part = ranking_df.loc[:, [f"{ticker}_Rank" for ticker in tickers]]
pos_part = strat_df.loc[:, [f"{ticker}_Position" for ticker in tickers]]
ret_part = bnh_df.loc[:, tickers].copy()
ret_part.columns = [f"{ticker}_Return" for ticker in tickers]

try:
    rank_part = rank_part.astype(float)
    pos_part = pos_part.astype(float)
    ret_part = ret_part.replace([np.inf, -np.inf], np.nan).astype(float)
except (TypeError, ValueError) as exc:
    raise TypeError(f"Rank, Position, and Buy & Hold returns must be numeric: {exc}") from exc

if not np.isfinite(rank_part.to_numpy(dtype=float)).all():
    raise ValueError("Rank contains non-finite values; cannot lag eligibility.")
rank_rounded = np.round(rank_part)
if not np.allclose(rank_part.to_numpy(dtype=float), rank_rounded, atol=1e-9):
    raise ValueError("Rank values must be whole numbers.")
rank_part = rank_rounded
if bool((rank_part < 0).any().any()) or bool((rank_part > len(tickers)).any().any()):
    raise ValueError(f"Rank values must be in 0..{len(tickers)}.")
positive_ranks = rank_part.where(rank_part > 0)
n_positive = positive_ranks.notna().sum(axis=1)
n_unique = positive_ranks.nunique(axis=1)
if bool((n_positive != n_unique).any()):
    raise ValueError("Duplicate positive ranks on some dates; top-N slots would overlap.")

if bool(pos_part.isna().any().any()):
    raise ValueError("Position contains missing values; cannot test position == 1.")
pos_ok = np.isclose(pos_part, 0.0) | np.isclose(pos_part, 1.0)
if not bool(pos_ok.all().all()):
    raise ValueError("Position values must be 0 or 1.")
pos_part = pos_part.where(~np.isclose(pos_part, 1.0), 1.0)
pos_part = pos_part.where(~np.isclose(pos_part, 0.0), 0.0)

combined_df = pd.concat([rank_part, pos_part, ret_part], axis=1)
if combined_df.columns.duplicated().any():
    dupes = combined_df.columns[combined_df.columns.duplicated()].tolist()
    raise ValueError(f"combined_df has duplicate column(s): {dupes}")
if len(combined_df) < 2:
    raise ValueError(
        f"Need at least 2 dates to lag ranks; got {len(combined_df)}."
    )

print("=" * 80)
print("STEP 1: PARAMETER VALIDATION")
print("=" * 80)
print(f"TOP_N: {TOP_N}")
print(f"SAFE_HAVEN: {SAFE_HAVEN}")
print(f"Max Weight per Asset: {1.0 / TOP_N:.2%}")
print(f"Available Tickers: {len(tickers)}")
print(", ".join(tickers))
print("-" * 80)
print("Each ticker has Rank, Position, and Buy & Hold Return columns.")
print(f"Date range: {combined_df.index.min()} to {combined_df.index.max()}")
print(f"Rows: {len(combined_df)}")
print("Step 1 Complete")

STEP 1: PARAMETER VALIDATION
TOP_N: 4
SAFE_HAVEN: GLD
Max Weight per Asset: 25.00%
Available Tickers: 6
QTUM, BTC-USD, TQQQ, SMH.L, UPRO, GLD
--------------------------------------------------------------------------------
Each ticker has Rank, Position, and Buy & Hold Return columns.
Date range: 2021-01-05 00:00:00 to 2026-10-07 00:00:00
Rows: 1446
Step 1 Complete


In [12]:
# STRATEGY EXECUTION: DYNAMIC ALLOCATION (RANK LAG + SESSION-START POSITION)
# Yesterday's rank and the position known at this ticker's session start decide today's weights.

if "combined_df" not in globals():
    raise NameError("combined_df is missing. Run the parameter-validation cell first.")
if "tickers" not in globals() or "TOP_N" not in globals() or "SAFE_HAVEN" not in globals():
    raise NameError("TOP_N / SAFE_HAVEN / tickers are missing. Run the parameter-validation cell first.")
if not isinstance(combined_df, pd.DataFrame) or combined_df.empty:
    raise ValueError("combined_df is missing or empty. Run the parameter-validation cell first.")
if combined_df.index.has_duplicates:
    raise ValueError("combined_df has duplicate dates; cannot allocate by date.")
if pd.isna(combined_df.index).any():
    raise ValueError("combined_df has missing dates; cannot allocate by date.")
if not combined_df.index.is_monotonic_increasing:
    raise ValueError("combined_df is not sorted by date; a 1-day lag would be wrong.")
if len(combined_df) < 2:
    raise ValueError("Need at least 2 dates so today's trade can use yesterday's rank.")
if not isinstance(tickers, (list, tuple, pd.Index)) or len(tickers) < 1:
    raise TypeError(f"tickers must be a non-empty sequence; got {type(tickers).__name__}")
tickers = [str(t) for t in tickers]
if len(set(tickers)) != len(tickers):
    raise ValueError(f"Duplicate ticker(s): {tickers}")

try:
    top_n_num = float(TOP_N)
except (TypeError, ValueError) as exc:
    raise ValueError(f"TOP_N must be numeric; got {TOP_N!r}") from exc
if not np.isfinite(top_n_num) or top_n_num != int(top_n_num):
    raise ValueError(f"TOP_N must be a whole number; got {TOP_N!r}")
TOP_N = int(top_n_num)
if TOP_N < 1:
    raise ValueError(f"TOP_N must be a positive integer; got {TOP_N!r}")
SAFE_HAVEN = str(SAFE_HAVEN).strip()
if SAFE_HAVEN not in tickers:
    raise KeyError(f"SAFE_HAVEN={SAFE_HAVEN!r} is not in tickers {tickers}.")

rank_cols = [f"{ticker}_Rank" for ticker in tickers]
pos_cols = [f"{ticker}_Position" for ticker in tickers]
weight_cols = [f"{ticker}_Weight" for ticker in tickers]
missing = [col for col in rank_cols + pos_cols if col not in combined_df.columns]
if missing:
    raise KeyError(f"combined_df is missing required column(s): {missing}")

if "raam_native_positions" not in globals() or not isinstance(raam_native_positions, dict):
    raise NameError("raam_native_positions is missing. Run the per-ticker file cell first.")
missing_native = [t for t in tickers if t not in raam_native_positions]
if missing_native:
    raise KeyError(f"No native position series for: {missing_native}")
if "_raam_session_start_cutoff" not in globals() or not callable(_raam_session_start_cutoff):
    raise NameError("_raam_session_start_cutoff is missing. Run the per-ticker file cell first.")
if "_raam_position_at_session_start" not in globals() or not callable(_raam_position_at_session_start):
    raise NameError("_raam_position_at_session_start is missing. Run the per-ticker file cell first.")

lagged_rank = combined_df.loc[:, rank_cols].astype(float).shift(1)
lagged_rank.columns = tickers
cutoff = _raam_session_start_cutoff(combined_df.index)
lagged_pos = pd.DataFrame(index=combined_df.index, columns=tickers, dtype=float)
for ticker in tickers:
    lagged_pos[ticker] = _raam_position_at_session_start(
        raam_native_positions[ticker], cutoff, combined_df.index, ticker
    )
started = lagged_pos.notna().cummax()
interior_nan = lagged_pos.isna() & started
if bool(interior_nan.any().any()):
    bad_cols = [col for col in lagged_pos.columns if bool(interior_nan[col].any())]
    raise ValueError(
        f"Interior missing session-start positions; cannot allocate: {bad_cols}"
    )
lagged_pos = lagged_pos.fillna(0.0)
pos_ok = np.isclose(lagged_pos, 0.0) | np.isclose(lagged_pos, 1.0)
if not bool(pos_ok.all().all()):
    raise ValueError("Session-start positions must be 0 or 1.")

# Dual eligibility: top-N rank from yesterday, position from this ticker's session start.
primary = (lagged_rank > 0) & (lagged_rank <= TOP_N) & lagged_pos.eq(1.0)
n_selected = primary.sum(axis=1)
if bool((n_selected > TOP_N).any()):
    raise ValueError(
        "More than TOP_N assets passed dual eligibility on some dates "
        "(duplicate ranks). Refusing to overweight the book."
    )
slots_needed = (TOP_N - n_selected).clip(lower=0)

# Leftover slots: SAFE_HAVEN needs rank > 0 and position == 1 (not rank <= TOP_N).
safe_haven_ok = (lagged_rank[SAFE_HAVEN] > 0) & lagged_pos[SAFE_HAVEN].eq(1.0)

weight_per_slot = 1.0 / TOP_N
weights = primary.astype(float) * weight_per_slot
extra = slots_needed.astype(float) * weight_per_slot
weights[SAFE_HAVEN] = weights[SAFE_HAVEN] + extra.where(safe_haven_ok, 0.0)
weights = weights.fillna(0.0)

allocation_df = weights.copy()
allocation_df.columns = weight_cols
allocation_df["Total_Weight"] = allocation_df[weight_cols].sum(axis=1)
if bool((allocation_df["Total_Weight"] > 1.0 + 1e-9).any()):
    raise ValueError(
        f"Total_Weight exceeds 100% (max={float(allocation_df['Total_Weight'].max()):.4%})."
    )
if not np.isclose(float(allocation_df["Total_Weight"].iloc[0]), 0.0):
    raise ValueError("Day-0 weights must be 0 after the rank lag / session-start position.")

held = weights.gt(0)
held_shift = held.shift(1, fill_value=False)
allocation_changed = (held != held_shift).any(axis=1)
allocation_changed.iloc[0] = False
allocation_history = combined_df.index[allocation_changed].tolist()

total = allocation_df["Total_Weight"]
cash_days = int(np.isclose(total, 0.0).sum())
full_days = int((total >= 1.0 - 1e-9).sum())
partial_days = int(len(allocation_df) - cash_days - full_days)
safe_haven_days = int((allocation_df[f"{SAFE_HAVEN}_Weight"] > 0).sum())

print("=" * 80)
print("STEP 2: DYNAMIC ASSET ALLOCATION (RANK LAG + SESSION-START POSITION)")
print("=" * 80)
print(f"Total Days: {len(allocation_df)}")
print(f"Days in Cash: {cash_days}")
print(f"Partial Allocation: {partial_days}")
print(f"Full Allocation: {full_days}")
print(f"Days with {SAFE_HAVEN}: {safe_haven_days}")
print(f"Allocation Changes: {len(allocation_history)}")
print("-" * 80)
print("Sample Weights (Last 10 Days):")
print(allocation_df[weight_cols + ["Total_Weight"]].tail(10))
print("Step 2 Complete")

STEP 2: DYNAMIC ASSET ALLOCATION (RANK LAG + SESSION-START POSITION)
Total Days: 1446
Days in Cash: 253
Partial Allocation: 297
Full Allocation: 896
Days with GLD: 547
Allocation Changes: 425
--------------------------------------------------------------------------------
Sample Weights (Last 10 Days):
            QTUM_Weight  BTC-USD_Weight  TQQQ_Weight  SMH.L_Weight  \
Date                                                                 
2026-09-24         0.25            0.25         0.25          0.25   
2026-09-25         0.25            0.25         0.25          0.25   
2026-09-28         0.25            0.25         0.25          0.25   
2026-09-29         0.25            0.25         0.25          0.25   
2026-09-30         0.25            0.25         0.25          0.25   
2026-10-01         0.25            0.25         0.25          0.25   
2026-10-02         0.25            0.25         0.25          0.25   
2026-10-05         0.25            0.25         0.25          0.25

LIVE TRADING - NEXT SESSION BUY / HOLD / SELL ACTIONS
----------------------------------------------------

This section turns the latest RAAM ranks and ensemble positions into a next-session action list.

- After the last bar closes, ranks and positions for that bar are known.
- Next-session target weights use those latest (unlagged) ranks and positions with the same dual-eligibility rules.
- Each ticker's target applies at the start of its own next session (europe / na / crypto).
- Current book = last lagged allocation row (what the strategy held on the last bar).
- BUY / SELL / HOLD / REBALANCE compare next-session targets to the current book.
- Leftover weight is cash when SAFE_HAVEN cannot fill empty slots.

---

In [13]:
# LIVE TRADING: NEXT-SESSION BUY / HOLD / SELL ACTIONS
# Latest ranks/positions -> next-session weights.
# Compare to the last lagged allocation (current strategy book).

if "allocation_df" not in globals():
    raise NameError("allocation_df is missing. Run the dynamic-allocation cell first.")
if "combined_df" not in globals():
    raise NameError("combined_df is missing. Run the parameter-validation cell first.")
if "tickers" not in globals() or "TOP_N" not in globals() or "SAFE_HAVEN" not in globals():
    raise NameError("TOP_N / SAFE_HAVEN / tickers are missing. Run the parameter-validation cell first.")
if not isinstance(allocation_df, pd.DataFrame) or allocation_df.empty:
    raise ValueError("allocation_df is missing or empty. Run the dynamic-allocation cell first.")
if not isinstance(combined_df, pd.DataFrame) or combined_df.empty:
    raise ValueError("combined_df is missing or empty. Run the parameter-validation cell first.")
if not combined_df.index.equals(allocation_df.index):
    raise ValueError("allocation_df and combined_df indexes do not match.")
if allocation_df.index.has_duplicates or combined_df.index.has_duplicates:
    raise ValueError("Duplicate dates found; cannot decide next-session actions.")
if pd.isna(allocation_df.index).any() or pd.isna(combined_df.index).any():
    raise ValueError("Missing dates found; cannot decide next-session actions.")
if not allocation_df.index.is_monotonic_increasing:
    raise ValueError("allocation_df is not sorted by date; cannot decide next-session actions.")
if len(allocation_df) < 2:
    raise ValueError("Need at least 2 dates so the lagged current book is defined.")
if not isinstance(tickers, (list, tuple, pd.Index)) or len(tickers) < 1:
    raise TypeError(f"tickers must be a non-empty sequence; got {type(tickers).__name__}")
tickers = [str(t) for t in tickers]
if len(set(tickers)) != len(tickers):
    raise ValueError(f"Duplicate ticker(s): {tickers}")

try:
    top_n_num = float(TOP_N)
except (TypeError, ValueError) as exc:
    raise ValueError(f"TOP_N must be numeric; got {TOP_N!r}") from exc
if not np.isfinite(top_n_num) or top_n_num != int(top_n_num) or int(top_n_num) < 1:
    raise ValueError(f"TOP_N must be a positive integer; got {TOP_N!r}")
TOP_N = int(top_n_num)
SAFE_HAVEN = str(SAFE_HAVEN).strip()
if SAFE_HAVEN not in tickers:
    raise KeyError(f"SAFE_HAVEN={SAFE_HAVEN!r} is not in tickers {tickers}.")

weight_cols = [f"{ticker}_Weight" for ticker in tickers]
rank_cols = [f"{ticker}_Rank" for ticker in tickers]
pos_cols = [f"{ticker}_Position" for ticker in tickers]
missing_w = [col for col in weight_cols if col not in allocation_df.columns]
missing_src = [col for col in rank_cols + pos_cols if col not in combined_df.columns]
if missing_w:
    raise KeyError(f"allocation_df is missing weight column(s): {missing_w}")
if missing_src:
    raise KeyError(f"combined_df is missing required column(s): {missing_src}")
if "Total_Weight" not in allocation_df.columns:
    raise KeyError("allocation_df is missing Total_Weight. Run the dynamic-allocation cell first.")

weight_mat = allocation_df.loc[:, weight_cols].astype(float)
weight_mat.columns = tickers
total = allocation_df["Total_Weight"].astype(float)
if bool((~np.isfinite(weight_mat.to_numpy(dtype=float))).any()) or bool((~np.isfinite(total)).any()):
    raise ValueError("Weights contain non-finite values.")
if bool((weight_mat < -1e-9).any().any()) or bool((total < -1e-9).any()):
    raise ValueError("Weights contain negative values.")
recomputed_total = weight_mat.sum(axis=1)
if not np.allclose(total.to_numpy(dtype=float), recomputed_total.to_numpy(dtype=float), atol=1e-9):
    raise ValueError("Total_Weight does not match the sum of asset weights.")
if bool((total > 1.0 + 1e-9).any()):
    raise ValueError(
        f"Total_Weight exceeds 100% (max={float(total.max()):.4%})."
    )

asof_date = allocation_df.index[-1]
current_weights = weight_mat.iloc[-1].astype(float)
current_total = float(total.iloc[-1])

# Position in force at this ticker's current/next session start.
# Do not use the aligned last equity date: crypto may already be in today's session.
if "raam_native_positions" not in globals() or not isinstance(raam_native_positions, dict):
    raise NameError("raam_native_positions is missing. Run the per-ticker file cell first.")
latest_rank = combined_df.loc[asof_date, rank_cols].astype(float)
latest_rank.index = tickers
latest_pos = pd.Series(index=tickers, dtype=float)
for ticker in tickers:
    if ticker not in raam_native_positions:
        raise KeyError(f"No native position series for {ticker!r}.")
    native_pos = pd.Series(raam_native_positions[ticker]).astype(float).dropna()
    if native_pos.empty:
        raise ValueError(f"Native position series is empty for {ticker!r}.")
    latest_pos[ticker] = float(native_pos.iloc[-1])

if bool((~np.isfinite(latest_rank.to_numpy(dtype=float))).any()):
    raise ValueError("Latest ranks contain non-finite values.")
if bool((~np.isfinite(latest_pos.to_numpy(dtype=float))).any()):
    raise ValueError("Latest positions contain non-finite values.")
rank_values = latest_rank.to_numpy(dtype=float)
rank_rounded = np.round(rank_values)
if not np.allclose(rank_values, rank_rounded, atol=1e-9):
    raise ValueError("Latest ranks must be whole numbers.")
latest_rank = pd.Series(rank_rounded, index=tickers, dtype=float)
if bool((latest_rank < 0).any()) or bool((latest_rank > len(tickers)).any()):
    raise ValueError(f"Latest ranks must be in 0..{len(tickers)}.")
pos_ok = np.isclose(latest_pos, 0.0) | np.isclose(latest_pos, 1.0)
if not bool(pos_ok.all()):
    raise ValueError("Latest positions must be 0 or 1.")
latest_pos = latest_pos.where(~np.isclose(latest_pos, 1.0), 1.0)
latest_pos = latest_pos.where(~np.isclose(latest_pos, 0.0), 0.0)

primary = (latest_rank > 0) & (latest_rank <= TOP_N) & latest_pos.eq(1.0)
n_selected = int(primary.sum())
if n_selected > TOP_N:
    raise ValueError(
        "More than TOP_N assets passed dual eligibility on the latest bar "
        "(duplicate ranks). Refusing to overweight the book."
    )
slots_needed = max(TOP_N - n_selected, 0)
safe_haven_ok = bool((latest_rank[SAFE_HAVEN] > 0) and np.isclose(latest_pos[SAFE_HAVEN], 1.0))

slot_weight = 1.0 / TOP_N
next_weights = primary.astype(float) * slot_weight
if safe_haven_ok and slots_needed > 0:
    next_weights[SAFE_HAVEN] = float(next_weights[SAFE_HAVEN]) + slots_needed * slot_weight
next_weights = next_weights.fillna(0.0).astype(float)
next_total = float(next_weights.sum())
if next_total > 1.0 + 1e-9:
    raise ValueError(f"Next-session Total_Weight exceeds 100% ({next_total:.4%}).")
if bool((next_weights < -1e-9).any()):
    raise ValueError("Next-session weights contain negative values.")
cash_weight = max(0.0, 1.0 - next_total)

# Stale / incomplete bar warnings (do not hard-fail; data vendors vary).
asof_ts = pd.Timestamp(asof_date).tz_localize(None) if getattr(asof_date, "tzinfo", None) else pd.Timestamp(asof_date)
today_ts = pd.Timestamp.today().normalize()
age_days = int((today_ts - asof_ts.normalize()).days)
stale_warnings = []
if age_days > 5:
    stale_warnings.append(
        f"Latest bar is {age_days} calendar days old ({asof_ts.date()}). "
        "Market data may be stale or unavailable."
    )
if asof_ts.normalize() == today_ts:
    stale_warnings.append(
        "Latest bar is dated today. If markets are still open, OHLC may be incomplete "
        "and next-session actions can change after the close."
    )

rows = []
for ticker in tickers:
    cur_w = float(current_weights[ticker])
    nxt_w = float(next_weights[ticker])
    if cur_w <= 1e-12 and nxt_w > 1e-12:
        action = "BUY"
    elif cur_w > 1e-12 and nxt_w <= 1e-12:
        action = "SELL"
    elif cur_w > 1e-12 and nxt_w > 1e-12:
        action = "REBALANCE" if abs(nxt_w - cur_w) > 1e-9 else "HOLD"
    else:
        action = "FLAT"
    rows.append({
        "Ticker": ticker,
        "Action": action,
        "Current_Weight": cur_w,
        "Next_Weight": nxt_w,
        "Weight_Change": nxt_w - cur_w,
        "Rank": int(latest_rank[ticker]),
        "Position": int(latest_pos[ticker]),
    })

actions_df = pd.DataFrame(rows)
if actions_df.empty:
    raise ValueError("No ticker actions were produced.")

buy_n = int((actions_df["Action"] == "BUY").sum())
hold_n = int((actions_df["Action"] == "HOLD").sum())
rebalance_n = int((actions_df["Action"] == "REBALANCE").sum())
sell_n = int((actions_df["Action"] == "SELL").sum())
flat_n = int((actions_df["Action"] == "FLAT").sum())

asof_label = asof_ts.date()

print("=" * 80)
print("LIVE TRADING: NEXT-SESSION ACTIONS")
print("=" * 80)
if "_raam_session_kind" not in globals() or not callable(_raam_session_kind):
    raise NameError("_raam_session_kind is missing. Run the per-ticker file cell first.")
print(f"Signals as of:     {asof_label}")
print(
    "Target session:    "
    + ", ".join(f"{t} {_raam_session_kind(t)}" for t in tickers)
)
print(f"TOP_N: {TOP_N}  |  slot weight: {slot_weight:.2%}  |  SAFE_HAVEN: {SAFE_HAVEN}")
print(f"Current invested:  {current_total:.2%}")
print(f"Next invested:     {next_total:.2%}  |  Next cash: {cash_weight:.2%}")
print("-" * 80)
print(
    f"BUY: {buy_n}  |  HOLD: {hold_n}  |  REBALANCE: {rebalance_n}  |  "
    f"SELL: {sell_n}  |  FLAT: {flat_n}"
)
if stale_warnings:
    print("-" * 80)
    for msg in stale_warnings:
        print(f"WARNING: {msg}")
print("-" * 80)

display_cols = [
    "Ticker", "Action", "Current_Weight", "Next_Weight", "Weight_Change", "Rank", "Position"
]
display_df = actions_df.loc[:, display_cols].copy()
for col in ("Current_Weight", "Next_Weight", "Weight_Change"):
    display_df[col] = display_df[col].map(lambda x: f"{float(x):.2%}")

print("Action list (assumes you currently hold the strategy's last-bar book):")
print(display_df.to_string(index=False))
print("-" * 80)

print("Next-session target portfolio:")
held = next_weights[next_weights > 1e-12].sort_values(ascending=False)
if held.empty:
    print("  100% CASH")
else:
    for ticker, weight in held.items():
        print(f"  {ticker:<12} {float(weight):.2%}")
    if cash_weight > 1e-9:
        print(f"  {'CASH':<12} {cash_weight:.2%}")

print("=" * 80)
print("Run after the market close so the latest bar is complete.")
print("=" * 80)

actions_df

LIVE TRADING: NEXT-SESSION ACTIONS
Signals as of:     2026-10-07
Target session:    QTUM na, BTC-USD crypto, TQQQ na, SMH.L europe, UPRO na, GLD na
TOP_N: 4  |  slot weight: 25.00%  |  SAFE_HAVEN: GLD
Current invested:  100.00%
Next invested:     75.00%  |  Next cash: 25.00%
--------------------------------------------------------------------------------
BUY: 0  |  HOLD: 3  |  REBALANCE: 0  |  SELL: 1  |  FLAT: 2
--------------------------------------------------------------------------------
Action list (assumes you currently hold the strategy's last-bar book):
 Ticker Action Current_Weight Next_Weight Weight_Change  Rank  Position
   QTUM   SELL         25.00%       0.00%       -25.00%     5         1
BTC-USD   HOLD         25.00%      25.00%         0.00%     1         1
   TQQQ   HOLD         25.00%      25.00%         0.00%     2         1
  SMH.L   HOLD         25.00%      25.00%         0.00%     3         1
   UPRO   FLAT          0.00%       0.00%         0.00%     4         0

,Ticker,Action,Current_Weight,Next_Weight,Weight_Change,Rank,Position
0,QTUM,SELL,0.25,0.00,-0.25,5,1
1,BTC-USD,HOLD,0.25,0.25,0.00,1,1
2,TQQQ,HOLD,0.25,0.25,0.00,2,1
3,SMH.L,HOLD,0.25,0.25,0.00,3,1
4,UPRO,FLAT,0.00,0.00,0.00,4,0
5,GLD,FLAT,0.00,0.00,0.00,0,0
